In [22]:
import pandas as pd

In [23]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"

headers = {
	"x-rapidapi-key": "086cd3f100msh34586ee679092e0p1eefcbjsn7ae15e1db669",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

live = response.json()

In [43]:
live

{'typeMatches': [{'matchType': 'Domestic',
   'seriesMatches': [{'seriesAdWrapper': {'seriesId': 10730,
      'seriesName': 'Plunket Shield 2025-26',
      'matches': [{'matchInfo': {'matchId': 132302,
         'seriesId': 10730,
         'seriesName': 'Plunket Shield 2025-26',
         'matchDesc': '17th Match',
         'matchFormat': 'TEST',
         'startDate': '1773005400000',
         'endDate': '1773289800000',
         'state': 'Stumps',
         'status': 'Day 1: Stumps - Otago trail by 158 runs',
         'team1': {'teamId': 186,
          'teamName': 'Auckland',
          'teamSName': 'AKL',
          'imageId': 172246},
         'team2': {'teamId': 105,
          'teamName': 'Otago',
          'teamSName': 'OTG',
          'imageId': 172197},
         'venueInfo': {'id': 418,
          'ground': 'Eden Park Outer Oval',
          'city': 'Auckland',
          'timezone': '+13:00',
          'latitude': '-36.874039',
          'longitude': '174.745636'},
         'currBatTea

In [37]:
print(type(live))

<class 'dict'>


In [38]:
print(live.keys())

dict_keys(['message'])


In [50]:
type_matches = live['typeMatches']
type_matches

[{'matchType': 'Domestic',
  'seriesMatches': [{'seriesAdWrapper': {'seriesId': 10730,
     'seriesName': 'Plunket Shield 2025-26',
     'matches': [{'matchInfo': {'matchId': 132302,
        'seriesId': 10730,
        'seriesName': 'Plunket Shield 2025-26',
        'matchDesc': '17th Match',
        'matchFormat': 'TEST',
        'startDate': '1773005400000',
        'endDate': '1773289800000',
        'state': 'Stumps',
        'status': 'Day 1: Stumps - Otago trail by 158 runs',
        'team1': {'teamId': 186,
         'teamName': 'Auckland',
         'teamSName': 'AKL',
         'imageId': 172246},
        'team2': {'teamId': 105,
         'teamName': 'Otago',
         'teamSName': 'OTG',
         'imageId': 172197},
        'venueInfo': {'id': 418,
         'ground': 'Eden Park Outer Oval',
         'city': 'Auckland',
         'timezone': '+13:00',
         'latitude': '-36.874039',
         'longitude': '174.745636'},
        'currBatTeamId': 105,
        'seriesStartDt': '17634

In [51]:

print(type_matches[0])


{'matchType': 'Domestic', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 10730, 'seriesName': 'Plunket Shield 2025-26', 'matches': [{'matchInfo': {'matchId': 132302, 'seriesId': 10730, 'seriesName': 'Plunket Shield 2025-26', 'matchDesc': '17th Match', 'matchFormat': 'TEST', 'startDate': '1773005400000', 'endDate': '1773289800000', 'state': 'Stumps', 'status': 'Day 1: Stumps - Otago trail by 158 runs', 'team1': {'teamId': 186, 'teamName': 'Auckland', 'teamSName': 'AKL', 'imageId': 172246}, 'team2': {'teamId': 105, 'teamName': 'Otago', 'teamSName': 'OTG', 'imageId': 172197}, 'venueInfo': {'id': 418, 'ground': 'Eden Park Outer Oval', 'city': 'Auckland', 'timezone': '+13:00', 'latitude': '-36.874039', 'longitude': '174.745636'}, 'currBatTeamId': 105, 'seriesStartDt': '1763424000000', 'seriesEndDt': '1775001600000', 'isTimeAnnounced': True, 'stateTitle': 'Stumps'}, 'matchScore': {'team1Score': {'inngs1': {'inningsId': 1, 'runs': 228, 'wickets': 10, 'overs': 71.1}}, 'team2Score': {'inngs

In [52]:
insert_query = """
INSERT INTO match_live (
    match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team2_id,
    team1, team2,
    venue, venue_id, venue_city,
    start_date, status
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
status = VALUES(status),
start_date = VALUES(start_date)
"""

In [59]:
#live_match
from datetime import datetime

matches = []

for type_match in live.get("typeMatches", []):
    for series_match in type_match.get("seriesMatches", []):

        series_data = series_match.get("seriesAdWrapper", {})

        series_id = series_data.get("seriesId")
        series_name = series_data.get("seriesName")

        for match in series_data.get("matches", []):

            info = match.get("matchInfo", {})

            raw_start = info.get("startDate")

            start_date = None
            if raw_start:
                start_date = datetime.fromtimestamp(int(raw_start)/1000)

            matches.append({

                "match_id": info.get("matchId"),

                "series_id": series_id,
                "series_name": series_name,

                "match_desc": info.get("matchDesc"),
                "match_format": info.get("matchFormat"),

                "team1_id": info.get("team1",{}).get("teamId"),
                "team2_id": info.get("team2",{}).get("teamId"),

                "team1": info.get("team1",{}).get("teamName"),
                "team2": info.get("team2",{}).get("teamName"),

                "venue_id": info.get("venueInfo",{}).get("id"),
                "venue": info.get("venueInfo",{}).get("ground"),
                "venue_city": info.get("venueInfo",{}).get("city"),

                "start_date": start_date,
                "status": info.get("status")

            })

In [60]:
from utils.db_connection import get_connection

conn = get_connection()
cursor = conn.cursor()

insert_query = """
INSERT INTO match_live (
    match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team2_id,
    team1, team2,
    venue_id, venue, venue_city,
    start_date, status
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
status = VALUES(status)
"""

data = []

for m in matches:
    data.append((
        m["match_id"],
        m["series_id"],
        m["series_name"],
        m["match_desc"],
        m["match_format"],
        m["team1_id"],
        m["team2_id"],
        m["team1"],
        m["team2"],
        m["venue_id"],
        m["venue"],
        m["venue_city"],
        m["start_date"],
        m["status"]
    ))

cursor.executemany(insert_query, data)

conn.commit()

cursor.close()
conn.close()

In [58]:
print(matches.columns.tolist())

AttributeError: 'list' object has no attribute 'columns'

In [ ]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install tabulate


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from tabulate import tabulate

In [ ]:
import pandas as pd

In [ ]:
from datetime import datetime

In [ ]:
scorecard_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

In [ ]:
import requests

scorecard_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

headers = {
	"x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}
print("Final URL:", scorecard_url)
response = requests.get(scorecard_url, headers=headers)

print(response.status_code)
print(response.json())
scorecard = response.json()

Final URL: https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/147359/scard
200
{'scorecard': [{'inningsid': 1, 'batsman': [{'id': 1431603, 'balls': 71, 'runs': 53, 'fours': 8, 'sixes': 0, 'strkrate': '74.65', 'name': 'Zakir Kathrada', 'nickname': 'Zakir Kathrada', 'iscaptain': False, 'iskeeper': True, 'outdec': 'c and b Ludwig Kaestner', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 21721, 'balls': 25, 'runs': 12, 'fours': 1, 'sixes': 0, 'strkrate': '48', 'name': 'Musawenkosi Twala', 'nickname': 'Musawenkosi Twala', 'iscaptain': False, 'iskeeper': False, 'outdec': 'b Lesego Kokohlabana', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 11009, 'balls': 5

In [ ]:
insert_batting_query = """
INSERT INTO batting_scorecard (
 innings_id, player_id, player_name,
 runs, balls, fours, sixes, strike_rate,
 dismissal, is_captain, is_keeper
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
balls = VALUES(balls),
strike_rate = VALUES(strike_rate)
"""

insert_bowling_query = """
INSERT INTO bowling_scorecard (
innings_id, player_id, player_name,
overs, maidens, runs, wickets, economy
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
wickets = VALUES(wickets),
runs = VALUES(runs),
economy = VALUES(economy);
"""


In [ ]:
from utils.db_connection import get_connection

conn = get_connection()
cursor = conn.cursor()

for innings in scorecard.get("scorecard", []):

    innings_id = innings.get("inningsid")

    # BATSMAN 
    for batsman in innings.get("batsman", []):

        batting_values = (
            innings_id,
            batsman.get("id"),
            batsman.get("name"),
            batsman.get("runs"),
            batsman.get("balls"),
            batsman.get("fours"),
            batsman.get("sixes"),
            batsman.get("strkrate"),
            batsman.get("outdec"),
            batsman.get("iscaptain"),
            batsman.get("iskeeper")
        )

        cursor.execute(insert_batting_query, batting_values)

    #  BOWLER
    for bowler in innings.get("bowler", []):

        bowling_values = (
            innings_id,
            bowler.get("id"),
            bowler.get("name"),
            bowler.get("overs"),
            bowler.get("maidens"),
            bowler.get("runs"),
            bowler.get("wickets"),
            bowler.get("economy")
        )

        cursor.execute(insert_bowling_query, bowling_values)

conn.commit()
cursor.close()
conn.close()

In [ ]:
scorecard = requests.get(scorecard_url).json()

for innings in scorecard.get("scorecard", []):

    innings_id = innings.get("inningsId")
    batting_team = innings.get("batTeamDetails", {}).get("batTeamName")

    runs = innings.get("scoreDetails", {}).get("runs")
    wickets = innings.get("scoreDetails", {}).get("wickets")
    overs = innings.get("scoreDetails", {}).get("overs")

    # Score
    mycursor.execute(insert_score_query, (
        match_id,
        innings_id,
        batting_team,
        runs,
        wickets,
        overs
    ))

    #  Batting
    for batsman in innings.get("batTeamDetails", {}).get("batsmenData", {}).values():

        player_name = batsman.get("batName")
        bat_runs = batsman.get("runs")
        balls = batsman.get("balls")
        fours = batsman.get("fours")
        sixes = batsman.get("sixes")
        sr = batsman.get("strikeRate")

        mycursor.execute(insert_batting_query, (
            match_id,
            innings_id,
            player_name,
            bat_runs,
            balls,
            fours,
            sixes,
            sr
        ))

    #  Bowling
    for bowler in innings.get("bowlTeamDetails", {}).get("bowlersData", {}).values():

        player_name = bowler.get("bowlName")
        bowl_overs = bowler.get("overs")
        bowl_runs = bowler.get("runs")
        bowl_wickets = bowler.get("wickets")
        eco = bowler.get("economy")

        mycursor.execute(insert_bowling_query, (
            match_id,
            innings_id,
            player_name,
            bowl_overs,
            bowl_runs,
            bowl_wickets,
            eco
        ))
circ.commit()


In [ ]:
circ.commit

<bound method CMySQLConnection.commit of <mysql.connector.connection_cext.CMySQLConnection object at 0x00000276D6C31950>>

In [4]:
import requests
import string
import pandas as pd

headers = {
    "x-rapidapi-key": "086cd3f100msh34586ee679092e0p1eefcbjsn7ae15e1db669",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

player_list = []

#  alphabet loop
for letter in string.ascii_lowercase:
    
    url = "https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/search"
    params = {"plrN": letter}
    
    response = requests.get(url, headers=headers, params=params)
    data = response.json()
    
    for player in data.get("player", []):
        
        player_list.append({
            "player_id": player.get("id"),
            "player_name": player.get("name")
        })

players_df = pd.DataFrame(player_list)

# duplicate removed
players_df = players_df.drop_duplicates(subset="player_id")

print(players_df.head())
print("Total players collected:", len(players_df))

KeyboardInterrupt: 

In [ ]:
if "scoreCard" in scorecard and scorecard["scoreCard"]:
    print("Last Innings ID:", innings_id)
else:
    print("No innings data found")

No innings data found


In [ ]:
print(scorecard.get("status"))
print(scorecard.get("scoreCard"))

None
None


In [ ]:
print("LIVE TEST:")
live_url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"
live_response = requests.get(live_url, headers=headers)
print(live_response.json())

print("\nSCORECARD TEST:")
scorecard_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/139461/scard"
score_response = requests.get(scorecard_url, headers=headers)
print(score_response.json())

LIVE TEST:
{'typeMatches': [{'matchType': 'Domestic', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 11553, 'seriesName': 'CSA Provincial One-Day Challenge Division One 2026', 'matches': [{'matchInfo': {'matchId': 147112, 'seriesId': 11553, 'seriesName': 'CSA Provincial One-Day Challenge Division One 2026', 'matchDesc': '7th Match', 'matchFormat': 'ODI', 'startDate': '1772622000000', 'endDate': '1772650800000', 'state': 'Toss', 'status': 'North West - Eastvaal Renault Dragons opt to bowl', 'team1': {'teamId': 157, 'teamName': 'Lions', 'teamSName': 'LIONS', 'imageId': 172224}, 'team2': {'teamId': 922, 'teamName': 'North West - Eastvaal Renault Dragons', 'teamSName': 'NWEST', 'imageId': 248439}, 'venueInfo': {'id': 102, 'ground': 'Senwes Park', 'city': 'Potchefstroom', 'timezone': '+02:00', 'latitude': '-26.695565', 'longitude': '27.100691'}, 'seriesStartDt': '1772150400000', 'seriesEndDt': '1774915200000', 'isTimeAnnounced': True, 'stateTitle': 'Toss'}}, {'matchInfo': {'matchId': 14

In [ ]:
print(scorecard)

{'message': 'Too many requests'}


In [ ]:
print(scorecard)
print(response.status_code)

{'message': 'Too many requests'}
200


In [ ]:
print(scorecard.keys())

dict_keys(['message'])


In [ ]:
print(scorecard.keys())

dict_keys(['message'])


In [ ]:
live_url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"

In [ ]:
import json
print(json.dumps(scorecard, indent=2))

{
  "message": "Too many requests"
}


In [ ]:
import requests

headers = {
    "X-RapidAPI-Key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
    "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
}

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.json())

200
{'typeMatches': [{'matchType': 'Domestic', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 11553, 'seriesName': 'CSA Provincial One-Day Challenge Division One 2026', 'matches': [{'matchInfo': {'matchId': 147104, 'seriesId': 11553, 'seriesName': 'CSA Provincial One-Day Challenge Division One 2026', 'matchDesc': '6th Match', 'matchFormat': 'ODI', 'startDate': '1772622000000', 'endDate': '1772650800000', 'state': 'Toss', 'status': 'Boland opt to bat', 'team1': {'teamId': 1759, 'teamName': 'Boland', 'teamSName': 'BOL', 'imageId': 349610}, 'team2': {'teamId': 1766, 'teamName': 'KwaZulu-Natal Inland - Tuskers', 'teamSName': 'KZNIN', 'imageId': 349611}, 'venueInfo': {'id': 207, 'ground': 'Boland Park', 'city': 'Paarl', 'timezone': '+02:00', 'latitude': '-33.730900', 'longitude': '18.970760'}, 'seriesStartDt': '1772150400000', 'seriesEndDt': '1774915200000', 'isTimeAnnounced': True, 'stateTitle': 'Toss'}}, {'matchInfo': {'matchId': 147112, 'seriesId': 11553, 'seriesName': 'CSA Provincia

In [ ]:
response = requests.get(url, headers=headers)
data = response.json()


In [ ]:
live_data = response.json()

for type_match in live_data.get("typeMatches", []):
    for series_match in type_match.get("seriesMatches", []):
        series = series_match.get("seriesAdWrapper", {})
        for match in series.get("matches", []):
            match_id = match.get("matchInfo", {}).get("matchId")
            print("Found match_id:", match_id)
            break

Found match_id: 147104
Found match_id: 147310


In [ ]:
scorecard_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{139461}/scard"

response = requests.get(scorecard_url, headers=headers)

print(response.status_code)
print(response.json())

200
{'scorecard': [{'inningsid': 1, 'batsman': [{'id': 10384, 'balls': 33, 'runs': 32, 'fours': 3, 'sixes': 1, 'strkrate': '96.97', 'name': 'Shai Hope', 'nickname': 'Shai Hope', 'iscaptain': True, 'iskeeper': True, 'outdec': 'b Varun Chakaravarthy', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 11543, 'balls': 25, 'runs': 40, 'fours': 5, 'sixes': 1, 'strkrate': '160', 'name': 'Roston Chase', 'nickname': 'Roston Chase', 'iscaptain': False, 'iskeeper': False, 'outdec': 'c Suryakumar Yadav b Jasprit Bumrah', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 9789, 'balls': 12, 'runs': 27, 'fours': 1, 'sixes': 2, 'strkrate': '225', 'name': 'Shimron Hetmyer', 

In [ ]:
import mysql.connector

circ = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sunday@123",
    database="cricbuzz_live"
)

In [ ]:
mycursor = circ.cursor(buffered=True)

In [ ]:
mycursor.execute("SHOW TABLES;")
print("Tables:", mycursor.fetchall())

Tables: [('batting_stats',), ('bowling_stats',), ('icc_batsman_rankings',), ('live_score_fact',), ('match_live',), ('recent_matches',), ('trending_players',)]


In [ ]:
import mysql.connector

circ = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sunday@123",
    database="cricbuzz_live"
)

In [ ]:
mycursor = circ.cursor(buffered=True)

In [ ]:
mycursor.execute("SELECT COUNT(*) FROM batting_stats;")
print("Batting rows:", mycursor.fetchone())

mycursor.execute("SELECT COUNT(*) FROM bowling_stats;")
print("Bowling rows:", mycursor.fetchone())

Batting rows: (22,)
Bowling rows: (13,)


In [ ]:
mycursor.execute("SELECT COUNT(*) FROM match_live;")
print(mycursor.fetchone())

(20,)


In [ ]:
mycursor = circ.cursor()
mycursor.execute("SELECT COUNT(*) FROM match_live;")
print("Total rows:", mycursor.fetchone())

Total rows: (20,)


In [ ]:
from utils.db_connection import get_connection

In [ ]:
from datetime import datetime

live_url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"

response = requests.get(live_url, headers=headers)
live_data = response.json()

for type_match in live_data.get("typeMatches", []):
    for series_match in type_match.get("seriesMatches", []):
        series = series_match.get("seriesAdWrapper", {})

        series_id = series.get("seriesId")
        series_name = series.get("seriesName")

        for match in series.get("matches", []):
            info = match.get("matchInfo", {})

            # 🔹 Convert timestamp properly HERE
            raw_start = info.get("startDate")

            if raw_start:
                start_date = datetime.fromtimestamp(int(raw_start) / 1000)
            else:
                start_date = None

            mycursor.execute(insert_query, (
                info.get("matchId"),
                series_id,
                series_name,
                info.get("matchDesc"),
                info.get("matchFormat"),
                info.get("team1", {}).get("teamName"),
                info.get("team2", {}).get("teamName"),
                info.get("venueInfo", {}).get("ground"),
                start_date,   # ✅ correct place
                info.get("status")
            ))

circ.commit()
print("Live matches inserted successfully")

Live matches inserted successfully


In [ ]:
mycursor.execute("SELECT match_id FROM match_live LIMIT 1;")
match_id = mycursor.fetchone()[0]
print("Using match_id:", match_id)

Using match_id: 122198


In [ ]:
tables = ["match_live", "batting_stats", "bowling_stats", "live_score_fact"]

for table in tables:
    mycursor.execute(f"SELECT COUNT(*) FROM {table};")
    print(table, ":", mycursor.fetchone())

match_live : (20,)
batting_stats : (22,)
bowling_stats : (13,)
live_score_fact : (2,)


In [ ]:
scorecard_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{122198}/scard"

response = requests.get(scorecard_url, headers=headers)
scorecard = response.json()

insert_score_query = """
INSERT INTO live_score_fact
(match_id, innings, batting_team, runs, wickets, overs)
VALUES (%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
wickets = VALUES(wickets),
overs = VALUES(overs)
"""


insert_batting_query = """
INSERT INTO batting_stats
(match_id, innings, player_name, runs, balls, fours, sixes, strike_rate)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
balls = VALUES(balls),
fours = VALUES(fours),
sixes = VALUES(sixes),
strike_rate = VALUES(strike_rate)
"""

insert_bowling_query = """
INSERT INTO bowling_stats
(match_id, innings, player_name, overs, runs, wickets, economy)
VALUES (%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
overs = VALUES(overs),
runs = VALUES(runs),
wickets = VALUES(wickets),
economy = VALUES(economy)
"""


for innings in scorecard.get("scorecard", []):

    innings_id = innings.get("inningsid")

    batsmen = innings.get("batsman", [])

    #  Calculate total runs
    total_runs = sum(int(b.get("runs", 0)) for b in batsmen)

    #   wickets 
    total_wickets = sum(1 for b in batsmen if b.get("outdec"))

    #  Overs 
    bowlers = innings.get("bowler", [])
    overs = bowlers[0].get("overs") if bowlers else None

    # Insert into live_score_fact
    mycursor.execute(insert_score_query, (
        match_id,
        innings_id,
        "Team",   # we don’t have team name in this structure
        total_runs,
        total_wickets,
        overs
    ))


    #  Batting
    for batsman in innings.get("batsman", []):

        mycursor.execute(insert_batting_query, (
            match_id,
            innings_id,
            batsman.get("name"),
            batsman.get("runs"),
            batsman.get("balls"),
            batsman.get("fours"),
            batsman.get("sixes"),
            batsman.get("strkrate")
        ))

    #  Bowling
    for bowler in innings.get("bowler", []):

        mycursor.execute(insert_bowling_query, (
            match_id,
            innings_id,
            bowler.get("name"),
            bowler.get("overs"),
            bowler.get("runs"),
            bowler.get("wickets"),
            bowler.get("econ")
        ))

circ.commit()
print("Scorecard inserted successfully")

Scorecard inserted successfully


In [ ]:
mycursor.execute("SELECT COUNT(*) FROM batting_stats;")
print("batting_stats:", mycursor.fetchone())

mycursor.execute("SELECT COUNT(*) FROM bowling_stats;")
print("bowling_stats:", mycursor.fetchone())

batting_stats: (22,)
bowling_stats: (13,)


In [21]:
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/recent"

headers = {
	"x-rapidapi-key": "f3b2ba43b4mshce68ad6a5f2a2d7p1a3701jsn0da633c136df",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

print(response.json())

{'typeMatches': [{'matchType': 'International', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 11253, 'seriesName': "ICC Men's T20 World Cup 2026", 'matches': [{'matchInfo': {'matchId': 139489, 'seriesId': 11253, 'seriesName': "ICC Men's T20 World Cup 2026", 'matchDesc': 'Final', 'matchFormat': 'T20', 'startDate': '1772976600000', 'endDate': '1772989200000', 'state': 'Complete', 'status': 'India won by 96 runs', 'team1': {'teamId': 2, 'teamName': 'India', 'teamSName': 'IND', 'imageId': 776162}, 'team2': {'teamId': 13, 'teamName': 'New Zealand', 'teamSName': 'NZ', 'imageId': 776333}, 'venueInfo': {'id': 50, 'ground': 'Narendra Modi Stadium', 'city': 'Ahmedabad', 'timezone': '+05:30', 'latitude': '23.091785', 'longitude': '72.597465'}, 'currBatTeamId': 2, 'seriesStartDt': '1770422400000', 'seriesEndDt': '1773100800000', 'isTimeAnnounced': True, 'stateTitle': 'IND Won', 'isFantasyEnabled': True}, 'matchScore': {'team1Score': {'inngs1': {'inningsId': 1, 'runs': 255, 'wickets': 5, 'over

In [22]:
recent = response.json()

In [23]:
recent

{'typeMatches': [{'matchType': 'International',
   'seriesMatches': [{'seriesAdWrapper': {'seriesId': 11253,
      'seriesName': "ICC Men's T20 World Cup 2026",
      'matches': [{'matchInfo': {'matchId': 139489,
         'seriesId': 11253,
         'seriesName': "ICC Men's T20 World Cup 2026",
         'matchDesc': 'Final',
         'matchFormat': 'T20',
         'startDate': '1772976600000',
         'endDate': '1772989200000',
         'state': 'Complete',
         'status': 'India won by 96 runs',
         'team1': {'teamId': 2,
          'teamName': 'India',
          'teamSName': 'IND',
          'imageId': 776162},
         'team2': {'teamId': 13,
          'teamName': 'New Zealand',
          'teamSName': 'NZ',
          'imageId': 776333},
         'venueInfo': {'id': 50,
          'ground': 'Narendra Modi Stadium',
          'city': 'Ahmedabad',
          'timezone': '+05:30',
          'latitude': '23.091785',
          'longitude': '72.597465'},
         'currBatTeamId': 2,

In [ ]:
import pandas as pd

In [11]:
from datetime import datetime

raw_start = info.get("startDate")

try:
    start_date = datetime.fromtimestamp(int(raw_start)/1000) if raw_start else None
except (ValueError, TypeError):
    start_date = None

In [47]:
insert_recent_query = """
INSERT INTO recent_matches (
   match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team2_id,
    team1, team2,
    venue_id, venue, venue_city,
    start_date, result
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
team1_id = VALUES(team1_id),
team2_id = VALUES(team2_id),
team1 = VALUES(team1),
team2 = VALUES(team2),
venue_id = VALUES(venue_id),
venue = VALUES(venue),
venue_city = VALUES(venue_city),
result = VALUES(result),
start_date = VALUES(start_date)
"""

In [48]:
from utils.db_connection import get_connection
from datetime import datetime

recent_matches = []
conn = get_connection()
cursor = conn.cursor()

for type_match in recent.get("typeMatches", []):
    for series_match in type_match.get("seriesMatches", []):
        series_data = series_match.get("seriesAdWrapper", {})
        
        if not series_data:
            continue
        
        series_id = series_data.get("seriesId")
        series_name = series_data.get("seriesName")

        for match in series_data.get("matches", []):
            info = match.get("matchInfo", {})

            recent_matches.append({
                "series_id": series_id,
                "series_name": series_name,
                "match_id": int(info.get("matchId")) if info.get("matchId") else None,
                "match_Desc": info.get("matchDesc"),
                "match_format": info.get("matchFormat"),

                "team1_id": info.get("team1", {}).get("teamId"),
                "team2_id": info.get("team2", {}).get("teamId"),

                "team1": info.get("team1", {}).get("teamName"),
                "team2": info.get("team2", {}).get("teamName"),
                
                "venue_id": info.get("venueInfo",{}).get("id"),
                "venue": info.get("venueInfo",{}).get("ground"),
                "venue_city": info.get("venueInfo",{}).get("city"),
                
                "start_date": start_date,
                "result": info.get("status"),
               
            })

print("Total matches extracted:", len(recent_matches))
conn.commit()
cursor.close()
conn.close()


Total matches extracted: 37


In [ ]:

conn = get_connection()
cursor = conn.cursor()

data = [
    (
        m["match_id"],
        m["series_id"],
        m["series_name"],
        m["match_Desc"],
        m["match_format"],
        
        m["team1_id"],
        m["team2_id"],

        m["team1"],
        m["team2"],

        m["venue_id"],
        m["venue"],
        m["venue_city"],
        
        m["start_date"],
        m["result"]
    )
    for m in recent_matches
]

cursor.executemany(insert_query, data)
conn.commit()

print("Rows affected:", cursor.rowcount)

cursor.close()
conn.close()

Rows affected: 37


In [13]:
#match upcoming
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/upcoming"

headers = {
	"x-rapidapi-key": "64ac7cdbacmshd400f941246fd04p1bd841jsn25ce69f2b2e4",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

print(response.json())
upcome_match = response.json()
upcome_match

{'typeMatches': [{'matchType': 'International', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 11584, 'seriesName': 'Pakistan tour of Bangladesh, 2026', 'matches': [{'matchInfo': {'matchId': 147863, 'seriesId': 11584, 'seriesName': 'Pakistan tour of Bangladesh, 2026', 'matchDesc': '1st ODI', 'matchFormat': 'ODI', 'startDate': '1773216900000', 'endDate': '1773245700000', 'state': 'Preview', 'status': 'Match starts at Mar 11, 08:15 GMT', 'team1': {'teamId': 6, 'teamName': 'Bangladesh', 'teamSName': 'BAN', 'imageId': 776210}, 'team2': {'teamId': 3, 'teamName': 'Pakistan', 'teamSName': 'PAK', 'imageId': 776308}, 'venueInfo': {'id': 128, 'ground': 'Shere Bangla National Stadium', 'city': 'Dhaka', 'timezone': '+06:00', 'latitude': '23.806966', 'longitude': '90.363576'}, 'seriesStartDt': '1773187200000', 'seriesEndDt': '1773705600000', 'isTimeAnnounced': True, 'stateTitle': 'Preview'}}]}}]}, {'matchType': 'Domestic', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 10829, 'seriesName': 

{'typeMatches': [{'matchType': 'International',
   'seriesMatches': [{'seriesAdWrapper': {'seriesId': 11584,
      'seriesName': 'Pakistan tour of Bangladesh, 2026',
      'matches': [{'matchInfo': {'matchId': 147863,
         'seriesId': 11584,
         'seriesName': 'Pakistan tour of Bangladesh, 2026',
         'matchDesc': '1st ODI',
         'matchFormat': 'ODI',
         'startDate': '1773216900000',
         'endDate': '1773245700000',
         'state': 'Preview',
         'status': 'Match starts at Mar 11, 08:15 GMT',
         'team1': {'teamId': 6,
          'teamName': 'Bangladesh',
          'teamSName': 'BAN',
          'imageId': 776210},
         'team2': {'teamId': 3,
          'teamName': 'Pakistan',
          'teamSName': 'PAK',
          'imageId': 776308},
         'venueInfo': {'id': 128,
          'ground': 'Shere Bangla National Stadium',
          'city': 'Dhaka',
          'timezone': '+06:00',
          'latitude': '23.806966',
          'longitude': '90.363576'

In [ ]:
#upcoming matches
from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()
upcoming_matches = []

for type_match in upcome_match.get("typeMatches", []):

    for series in type_match.get("seriesMatches", []):

        wrapper = series.get("seriesAdWrapper")
        if not wrapper:
            continue

        for match in wrapper.get("matches", []):

            info = match.get("matchInfo", {})
            venue = info.get("venueInfo", {})

            raw_start = info.get("startDate")
            start_date = datetime.fromtimestamp(int(raw_start)/1000) if raw_start else None

            raw_end = info.get("endDate")
            end_date = datetime.fromtimestamp(int(raw_end)/1000) if raw_end else None

            upcoming_matches.append(
                (
                    info.get("matchId"),
                    info.get("seriesId"),
                    info.get("seriesName"),
                    info.get("matchDesc"),
                    info.get("matchFormat"),

                    info.get("team1", {}).get("teamId"),
                    info.get("team1", {}).get("teamName"),

                    info.get("team2", {}).get("teamId"),
                    info.get("team2", {}).get("teamName"),

                    venue.get("id"),
                    venue.get("ground"),
                    venue.get("city"),

                    start_date,
                    end_date,

                    info.get("state"),
                    info.get("status")
                )
            )


In [18]:
insert_query = """
INSERT INTO upcoming_matches (
    match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team1,
    team2_id, team2,
    venue_id, venue, venue_city,
    start_date, end_date,
    match_state, match_status
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
match_state = VALUES(match_state),
match_status = VALUES(match_status),
start_date = VALUES(start_date),
end_date = VALUES(end_date),
venue_id = VALUES(venue_id),
venue = VALUES(venue),
venue_city = VALUES(venue_city)
"""

conn = get_connection()
cursor = conn.cursor()

cursor.executemany(insert_query, upcoming_matches)

conn.commit()

print("Rows affected:", cursor.rowcount)

cursor.close()
conn.close()

Rows affected: 3


In [45]:
for row in data[:]:
    print("venue_id:", row[9], "venue:", row[10], "city:", row[11])

venue_id: 50 venue: Narendra Modi Stadium city: Ahmedabad
venue_id: 81 venue: Wankhede Stadium city: Mumbai
venue_id: 31 venue: Eden Gardens city: Kolkata
venue_id: 892 venue: Jimmy Powell Oval city: George Town
venue_id: 892 venue: Jimmy Powell Oval city: George Town
venue_id: 892 venue: Jimmy Powell Oval city: George Town
venue_id: 892 venue: Jimmy Powell Oval city: George Town
venue_id: 394 venue: Bayuemas Oval city: Kuala Lumpur
venue_id: 394 venue: Bayuemas Oval city: Kuala Lumpur
venue_id: 1305 venue: Botswana Cricket Association Oval 1 city: Gaborone
venue_id: 206 venue: City Oval city: Pietermaritzburg
venue_id: 75 venue: SuperSport Park city: Centurion
venue_id: 102 venue: Senwes Park city: Potchefstroom
venue_id: 88 venue: The Wanderers Stadium city: Johannesburg
venue_id: 74 venue: Kingsmead city: Durban
venue_id: 207 venue: Boland Park city: Paarl
venue_id: 102 venue: Senwes Park city: Potchefstroom
venue_id: 52 venue: Newlands city: Cape Town
venue_id: 300 venue: Allan Bor

In [46]:
for row in data:
    if row[9] is None or row[11] is None:
        print(row[0], row[9], row[11])

In [ ]:
import requests

icc_ranking_url = "https://cricbuzz-cricket.p.rapidapi.com/stats/v1/rankings/batsmen"

headers = {
    "x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

formats = ["test", "odi", "t20"]
all_rankings = {}

for fmt in formats:
    querystring = {"formatType": fmt}
    response = requests.get(icc_ranking_url, headers=headers, params=querystring)
    all_rankings[fmt] = response.json()

print(all_rankings)


{'test': {'rank': [{'id': '8019', 'rank': '1', 'name': 'Joe Root', 'country': 'England', 'rating': '880', 'points': '880', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '717792', 'countryId': '9'}, {'id': '12201', 'rank': '2', 'name': 'Harry Brook', 'country': 'England', 'rating': '857', 'points': '857', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '848570', 'countryId': '9'}, {'id': '8497', 'rank': '3', 'name': 'Travis Head', 'country': 'Australia', 'rating': '853', 'points': '853', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '845768', 'countryId': '4'}, {'id': '2250', 'rank': '4', 'name': 'Steven Smith', 'country': 'Australia', 'rating': '831', 'points': '831', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '619873', 'countryId': '4'}, {'id': '6326', 'rank': '5', 'name': 'Kane Williamson', 'country': 'New Zealand', 'rating': '822', 'points': '822', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '616

In [ ]:
import requests

icc_ranking_url = "https://cricbuzz-cricket.p.rapidapi.com/stats/v1/rankings/batsmen"

headers = {
    "x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

formats = ["test", "odi", "t20"]
all_rankings = {}

for fmt in formats:
    querystring = {"formatType": fmt}
    response = requests.get(icc_ranking_url, headers=headers, params=querystring)
    all_rankings[fmt] = response.json()

print(all_rankings)

{'test': {'rank': [{'id': '8019', 'rank': '1', 'name': 'Joe Root', 'country': 'England', 'rating': '880', 'points': '880', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '717792', 'countryId': '9'}, {'id': '12201', 'rank': '2', 'name': 'Harry Brook', 'country': 'England', 'rating': '857', 'points': '857', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '848570', 'countryId': '9'}, {'id': '8497', 'rank': '3', 'name': 'Travis Head', 'country': 'Australia', 'rating': '853', 'points': '853', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '845768', 'countryId': '4'}, {'id': '2250', 'rank': '4', 'name': 'Steven Smith', 'country': 'Australia', 'rating': '831', 'points': '831', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '619873', 'countryId': '4'}, {'id': '6326', 'rank': '5', 'name': 'Kane Williamson', 'country': 'New Zealand', 'rating': '822', 'points': '822', 'lastUpdatedOn': '2026-03-03', 'trend': 'Flat', 'faceImageId': '616

In [ ]:
all_rankings

{'test': {'rank': [{'id': '8019',
    'rank': '1',
    'name': 'Joe Root',
    'country': 'England',
    'rating': '880',
    'points': '880',
    'lastUpdatedOn': '2026-03-03',
    'trend': 'Flat',
    'faceImageId': '717792',
    'countryId': '9'},
   {'id': '12201',
    'rank': '2',
    'name': 'Harry Brook',
    'country': 'England',
    'rating': '857',
    'points': '857',
    'lastUpdatedOn': '2026-03-03',
    'trend': 'Flat',
    'faceImageId': '848570',
    'countryId': '9'},
   {'id': '8497',
    'rank': '3',
    'name': 'Travis Head',
    'country': 'Australia',
    'rating': '853',
    'points': '853',
    'lastUpdatedOn': '2026-03-03',
    'trend': 'Flat',
    'faceImageId': '845768',
    'countryId': '4'},
   {'id': '2250',
    'rank': '4',
    'name': 'Steven Smith',
    'country': 'Australia',
    'rating': '831',
    'points': '831',
    'lastUpdatedOn': '2026-03-03',
    'trend': 'Flat',
    'faceImageId': '619873',
    'countryId': '4'},
   {'id': '6326',
    'rank':

In [ ]:
import mysql.connector

circ = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sunday@123",
    database="cricbuzz_live"
)
mycursor = circ.cursor(buffered=True)

In [ ]:
insert_query = """
INSERT INTO icc_batsman_rankings
(player_id, player_name, country, format, ranking_position,
 rating, points, trend, last_updated, face_image_id, country_id)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
ranking_position=VALUES(ranking_position),
rating=VALUES(rating),
points=VALUES(points),
trend=VALUES(trend),
last_updated=VALUES(last_updated)
"""

for fmt, data in all_rankings.items():
    for player in data['rank']:
        mycursor.execute(insert_query, (
            int(player['id']),
            player['name'],
            player['country'],
            fmt.upper(),
            int(player['rank']),
            int(player['rating']),
            int(player['points']),
            player['trend'],
            player['lastUpdatedOn'],
            int(player['faceImageId']),
            int(player['countryId'])
        ))

circ.commit()
mycursor.close()
circ.close()

print(" ICC rankings inserted successfully!")

 ICC rankings inserted successfully!


In [ ]:
#trending players
import requests

trending_url = "https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/trending"

headers = {
	"x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(trending_url, headers=headers)

print(response.json())
trending = response.json()
trending

{'player': [{'id': '17613', 'name': 'Mohammed Aslam', 'teamName': 'Kuwait', 'faceImageId': 182026}, {'id': '8271', 'name': 'Sanju Samson', 'teamName': 'India', 'faceImageId': 846035}, {'id': '9354', 'name': 'Sikandar Raza', 'teamName': 'Zimbabwe', 'faceImageId': 848400}, {'id': '13625', 'name': 'Sahibzada Farhan', 'teamName': 'Pakistan', 'faceImageId': 846069}, {'id': '41715', 'name': 'Ali Dawood', 'teamName': 'Bahrain', 'faceImageId': 182026}, {'id': '27884', 'name': 'Thinley Jamtsho', 'teamName': 'Bhutan', 'faceImageId': 182026}, {'id': '12258', 'name': 'Will Jacks', 'teamName': 'England', 'faceImageId': 848529}, {'id': '34515', 'name': 'Sohail Ahmed', 'teamName': 'Bahrain', 'faceImageId': 182026}, {'id': '32595', 'name': 'Declan Suzuki', 'teamName': 'Japan', 'faceImageId': 182026}, {'id': '9647', 'name': 'Hardik Pandya', 'teamName': 'India', 'faceImageId': 846032}], 'category': 'TRENDING SEARCHES'}


{'player': [{'id': '17613',
   'name': 'Mohammed Aslam',
   'teamName': 'Kuwait',
   'faceImageId': 182026},
  {'id': '8271',
   'name': 'Sanju Samson',
   'teamName': 'India',
   'faceImageId': 846035},
  {'id': '9354',
   'name': 'Sikandar Raza',
   'teamName': 'Zimbabwe',
   'faceImageId': 848400},
  {'id': '13625',
   'name': 'Sahibzada Farhan',
   'teamName': 'Pakistan',
   'faceImageId': 846069},
  {'id': '41715',
   'name': 'Ali Dawood',
   'teamName': 'Bahrain',
   'faceImageId': 182026},
  {'id': '27884',
   'name': 'Thinley Jamtsho',
   'teamName': 'Bhutan',
   'faceImageId': 182026},
  {'id': '12258',
   'name': 'Will Jacks',
   'teamName': 'England',
   'faceImageId': 848529},
  {'id': '34515',
   'name': 'Sohail Ahmed',
   'teamName': 'Bahrain',
   'faceImageId': 182026},
  {'id': '32595',
   'name': 'Declan Suzuki',
   'teamName': 'Japan',
   'faceImageId': 182026},
  {'id': '9647',
   'name': 'Hardik Pandya',
   'teamName': 'India',
   'faceImageId': 846032}],
 'category

In [ ]:
trending

{'player': [{'id': '17613',
   'name': 'Mohammed Aslam',
   'teamName': 'Kuwait',
   'faceImageId': 182026},
  {'id': '8271',
   'name': 'Sanju Samson',
   'teamName': 'India',
   'faceImageId': 846035},
  {'id': '9354',
   'name': 'Sikandar Raza',
   'teamName': 'Zimbabwe',
   'faceImageId': 848400},
  {'id': '13625',
   'name': 'Sahibzada Farhan',
   'teamName': 'Pakistan',
   'faceImageId': 846069},
  {'id': '41715',
   'name': 'Ali Dawood',
   'teamName': 'Bahrain',
   'faceImageId': 182026},
  {'id': '27884',
   'name': 'Thinley Jamtsho',
   'teamName': 'Bhutan',
   'faceImageId': 182026},
  {'id': '12258',
   'name': 'Will Jacks',
   'teamName': 'England',
   'faceImageId': 848529},
  {'id': '34515',
   'name': 'Sohail Ahmed',
   'teamName': 'Bahrain',
   'faceImageId': 182026},
  {'id': '32595',
   'name': 'Declan Suzuki',
   'teamName': 'Japan',
   'faceImageId': 182026},
  {'id': '9647',
   'name': 'Hardik Pandya',
   'teamName': 'India',
   'faceImageId': 846032}],
 'category

In [ ]:

players = trending.get("player", [])

insert_query = """
INSERT INTO trending_players (
    player_id, player_name, team_name, face_image_id
)
VALUES (%s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
player_name = VALUES(player_name),
team_name = VALUES(team_name),
face_image_id = VALUES(face_image_id)
"""

for p in players:
    mycursor.execute(insert_query, (
        int(p.get("id")),
        p.get("name"),
        p.get("teamName"),
        p.get("faceImageId")
    ))

circ.commit()

mycursor.close()
circ.close()

print("Trending players inserted successfully")

Trending players inserted successfully


In [ ]:
tables = ["match_live", "batting_stats", "bowling_stats", "live_score_fact", "recent_matches", "icc_batsman_rankings","trending_players"]

for table in tables:
    mycursor.execute(f"SELECT COUNT(*) FROM {table};")
    count = mycursor.fetchone()[0]
    print(f"{table}: {count}")

match_live: 15
batting_stats: 22
bowling_stats: 13
live_score_fact: 2
recent_matches: 49
icc_batsman_rankings: 45
trending_players: 10


In [5]:
#schedule
import requests

schedule_url = "https://cricbuzz-cricket.p.rapidapi.com/schedule/v1/international"

headers = {
	"x-rapidapi-key": "64ac7cdbacmshd400f941246fd04p1bd841jsn25ce69f2b2e4",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(schedule_url, headers=headers)

print(response.json())
schedule_inter = response.json()
schedule_inter

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'WED, MAR 11 2026', 'matchScheduleList': [{'seriesName': 'Pakistan tour of Bangladesh, 2026', 'matchInfo': [{'matchId': 147863, 'seriesId': 11584, 'matchDesc': '1st ODI', 'matchFormat': 'ODI', 'startDate': '1773216900000', 'endDate': '1773273599000', 'team1': {'teamId': 6, 'teamName': 'Bangladesh', 'teamSName': 'BAN', 'isFullMember': True, 'imageId': 776210}, 'team2': {'teamId': 3, 'teamName': 'Pakistan', 'teamSName': 'PAK', 'isFullMember': True, 'imageId': 776308}, 'venueInfo': {'ground': 'Shere Bangla National Stadium', 'city': 'Dhaka', 'country': 'Bangladesh', 'timezone': '+06:00'}}], 'seriesId': 11584, 'seriesCategory': 'International'}, {'seriesName': "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026", 'matchInfo': [{'matchId': 147541, 'seriesId': 11570, 'matchDesc': '5th Match', 'matchFormat': 'T20', 'startDate': '1773239400000', 'endDate': '1773273599000', 'team1': {'teamId': 2285, 'teamName': 'Suriname', '

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'WED, MAR 11 2026',
    'matchScheduleList': [{'seriesName': 'Pakistan tour of Bangladesh, 2026',
      'matchInfo': [{'matchId': 147863,
        'seriesId': 11584,
        'matchDesc': '1st ODI',
        'matchFormat': 'ODI',
        'startDate': '1773216900000',
        'endDate': '1773273599000',
        'team1': {'teamId': 6,
         'teamName': 'Bangladesh',
         'teamSName': 'BAN',
         'isFullMember': True,
         'imageId': 776210},
        'team2': {'teamId': 3,
         'teamName': 'Pakistan',
         'teamSName': 'PAK',
         'isFullMember': True,
         'imageId': 776308},
        'venueInfo': {'ground': 'Shere Bangla National Stadium',
         'city': 'Dhaka',
         'country': 'Bangladesh',
         'timezone': '+06:00'}}],
      'seriesId': 11584,
      'seriesCategory': 'International'},
     {'seriesName': "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026",
      'matchInfo': [

In [6]:
print(schedule_inter.keys())

dict_keys(['matchScheduleMap', 'startDt', 'appIndex'])


In [ ]:

insert_query = """
INSERT INTO schedule_matches (
    match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team1,
    team2_id, team2,
    venue, venue_city, venue_country,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
team1_id = VALUES(team1_id),
team2_id = VALUES(team2_id),
team1 = VALUES(team1),
team2 = VALUES(team2),
venue = VALUES(venue),
venue_city = VALUES(venue_city),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

In [63]:
print(insert_query.count("%s"))

13


In [2]:
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/schedule/v1/league"

headers = {
	"x-rapidapi-key": "64ac7cdbacmshd400f941246fd04p1bd841jsn25ce69f2b2e4",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

print(response.json())
sch_league = response.json()
sch_league

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'WED, MAR 11 2026', 'matchScheduleList': [{'seriesName': 'Legends League Cricket 2026', 'matchInfo': [{'matchId': 148514, 'seriesId': 11671, 'matchDesc': '1st Match', 'matchFormat': 'T20', 'startDate': '1773237600000', 'endDate': '1773273599000', 'team1': {'teamId': 3160, 'teamName': 'Mumbai Spartans', 'teamSName': 'MS', 'imageId': 879555}, 'team2': {'teamId': 3182, 'teamName': 'India Captains', 'teamSName': 'INDCAP', 'imageId': 879552}, 'venueInfo': {'ground': 'Indira Gandhi International Cricket Stadium', 'city': 'Haldwani', 'country': 'India', 'timezone': '+05:30'}}], 'seriesId': 11671, 'seriesCategory': 'League'}], 'longDate': '1773187200000'}}, {'adDetail': {'name': 'native_news_index_random_1', 'layout': 'native_large', 'position': 1}}, {'adDetail': {'name': 'native_news_index_random_1', 'layout': 'native_large', 'position': 1}}, {'scheduleAdWrapper': {'date': 'THU, MAR 12 2026', 'matchScheduleList': [{'seriesName': 'Legends Le

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'WED, MAR 11 2026',
    'matchScheduleList': [{'seriesName': 'Legends League Cricket 2026',
      'matchInfo': [{'matchId': 148514,
        'seriesId': 11671,
        'matchDesc': '1st Match',
        'matchFormat': 'T20',
        'startDate': '1773237600000',
        'endDate': '1773273599000',
        'team1': {'teamId': 3160,
         'teamName': 'Mumbai Spartans',
         'teamSName': 'MS',
         'imageId': 879555},
        'team2': {'teamId': 3182,
         'teamName': 'India Captains',
         'teamSName': 'INDCAP',
         'imageId': 879552},
        'venueInfo': {'ground': 'Indira Gandhi International Cricket Stadium',
         'city': 'Haldwani',
         'country': 'India',
         'timezone': '+05:30'}}],
      'seriesId': 11671,
      'seriesCategory': 'League'}],
    'longDate': '1773187200000'}},
  {'adDetail': {'name': 'native_news_index_random_1',
    'layout': 'native_large',
    'position': 1}},
  {'adDetail'

In [3]:
insert_query = """
INSERT INTO schedule_matches (
    match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team1,
    team2_id, team2,
    venue, venue_city, venue_country,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
team1_id = VALUES(team1_id),
team2_id = VALUES(team2_id),
team1 = VALUES(team1),
team2 = VALUES(team2),
venue = VALUES(venue),
venue_city = VALUES(venue_city),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

In [4]:
#schedule league
from utils.db_connection import get_connection
from datetime import datetime

matches = []

for day in sch_league.get("matchScheduleMap", []):

    wrapper = day.get("scheduleAdWrapper")
    if not wrapper:
        continue

    for series in wrapper.get("matchScheduleList", []):

        series_name = series.get("seriesName")
        series_id = series.get("seriesId")

        for match in series.get("matchInfo", []):

            venue = match.get("venueInfo", {})

            raw_start = match.get("startDate")
            start_date = datetime.fromtimestamp(int(raw_start)/1000) if raw_start else None

            raw_end = match.get("endDate")
            end_date = datetime.fromtimestamp(int(raw_end)/1000) if raw_end else None

            matches.append(
                (
                    match.get("matchId"),
                    series_id,
                    series_name,
                    match.get("matchDesc"),
                    match.get("matchFormat"),

                    match.get("team1", {}).get("teamId"),
                    match.get("team1", {}).get("teamName"),

                    match.get("team2", {}).get("teamId"),
                    match.get("team2", {}).get("teamName"),

                    venue.get("ground"),
                    venue.get("city"),
                    venue.get("country"),

                    start_date,
                    end_date
                )
            )





conn = get_connection()
cursor = conn.cursor()

cursor.executemany(insert_query, matches)

conn.commit()

print("Inserted rows:", cursor.rowcount)

cursor.close()
conn.close()

cursor.close()
conn.close()

Inserted rows: 0


In [7]:
#schedule matches
from utils.db_connection import get_connection
from datetime import datetime



matches = []

for day in schedule_inter.get("matchScheduleMap", []):

    wrapper = day.get("scheduleAdWrapper")
    if not wrapper:
        continue

    for series in wrapper.get("matchScheduleList", []):

        series_name = series.get("seriesName")
        series_id = series.get("seriesId")

        for match in series.get("matchInfo", []):

            venue = match.get("venueInfo", {})

            raw_start = match.get("startDate")
            start_date = datetime.fromtimestamp(int(raw_start)/1000) if raw_start else None

            raw_end = match.get("endDate")
            end_date = datetime.fromtimestamp(int(raw_end)/1000) if raw_end else None

            matches.append(
                (
                    match.get("matchId"),
                    series_id,
                    series_name,
                    match.get("matchDesc"),
                    match.get("matchFormat"),

                    match.get("team1", {}).get("teamId"),
                    match.get("team1", {}).get("teamName"),

                    match.get("team2", {}).get("teamId"),
                    match.get("team2", {}).get("teamName"),

                    venue.get("ground"),
                    venue.get("city"),
                    venue.get("country"),

                    start_date,
                    end_date
                )
            )



conn = get_connection()
cursor = conn.cursor()

cursor.executemany(insert_query, matches)

conn.commit()

print("Inserted rows:", cursor.rowcount)

cursor.close()
conn.close()

cursor.close()
conn.close()

Inserted rows: 5


In [8]:
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/schedule/v1/domestic"

headers = {
	"x-rapidapi-key": "64ac7cdbacmshd400f941246fd04p1bd841jsn25ce69f2b2e4",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

print(response.json())
sch_domestic = response.json()
sch_domestic

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'TUE, MAR 10 2026', 'matchScheduleList': [{'seriesName': 'Plunket Shield 2025-26', 'matchInfo': [{'matchId': 132302, 'seriesId': 10730, 'matchDesc': '17th Match, Day 2', 'matchFormat': 'TEST', 'startDate': '1773091800000', 'endDate': '1773359999000', 'team1': {'teamId': 186, 'teamName': 'Auckland', 'teamSName': 'AKL', 'imageId': 172246}, 'team2': {'teamId': 105, 'teamName': 'Otago', 'teamSName': 'OTG', 'imageId': 172197}, 'venueInfo': {'ground': 'Eden Park Outer Oval', 'city': 'Auckland', 'country': 'New Zealand', 'timezone': '+13:00'}}, {'matchId': 132318, 'seriesId': 10730, 'matchDesc': '18th Match, Day 2', 'matchFormat': 'TEST', 'startDate': '1773091800000', 'endDate': '1773359999000', 'team1': {'teamId': 314, 'teamName': 'Wellington', 'teamSName': 'WEL', 'imageId': 172364}, 'team2': {'teamId': 312, 'teamName': 'Canterbury', 'teamSName': 'CNTBRY', 'imageId': 172362}, 'venueInfo': {'ground': 'Hagley Oval', 'city': 'Christchurch', '

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'TUE, MAR 10 2026',
    'matchScheduleList': [{'seriesName': 'Plunket Shield 2025-26',
      'matchInfo': [{'matchId': 132302,
        'seriesId': 10730,
        'matchDesc': '17th Match, Day 2',
        'matchFormat': 'TEST',
        'startDate': '1773091800000',
        'endDate': '1773359999000',
        'team1': {'teamId': 186,
         'teamName': 'Auckland',
         'teamSName': 'AKL',
         'imageId': 172246},
        'team2': {'teamId': 105,
         'teamName': 'Otago',
         'teamSName': 'OTG',
         'imageId': 172197},
        'venueInfo': {'ground': 'Eden Park Outer Oval',
         'city': 'Auckland',
         'country': 'New Zealand',
         'timezone': '+13:00'}},
       {'matchId': 132318,
        'seriesId': 10730,
        'matchDesc': '18th Match, Day 2',
        'matchFormat': 'TEST',
        'startDate': '1773091800000',
        'endDate': '1773359999000',
        'team1': {'teamId': 314,
         'team

In [9]:
insert_query = """
INSERT INTO schedule_matches (
    match_id, series_id, series_name,
    match_desc, match_format,
    team1_id, team1,
    team2_id, team2,
    venue, venue_city, venue_country,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
team1_id = VALUES(team1_id),
team2_id = VALUES(team2_id),
team1 = VALUES(team1),
team2 = VALUES(team2),
venue = VALUES(venue),
venue_city = VALUES(venue_city),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

In [10]:
#schedule domestic
from utils.db_connection import get_connection
from datetime import datetime



matches = []

for day in sch_domestic.get("matchScheduleMap", []):

    wrapper = day.get("scheduleAdWrapper")
    if not wrapper:
        continue

    for series in wrapper.get("matchScheduleList", []):

        series_name = series.get("seriesName")
        series_id = series.get("seriesId")

        for match in series.get("matchInfo", []):

            venue = match.get("venueInfo", {})

            raw_start = match.get("startDate")
            start_date = datetime.fromtimestamp(int(raw_start)/1000) if raw_start else None

            raw_end = match.get("endDate")
            end_date = datetime.fromtimestamp(int(raw_end)/1000) if raw_end else None

            matches.append(
                (
                    match.get("matchId"),
                    series_id,
                    series_name,
                    match.get("matchDesc"),
                    match.get("matchFormat"),

                    match.get("team1", {}).get("teamId"),
                    match.get("team1", {}).get("teamName"),

                    match.get("team2", {}).get("teamId"),
                    match.get("team2", {}).get("teamName"),

                    venue.get("ground"),
                    venue.get("city"),
                    venue.get("country"),

                    start_date,
                    end_date
                )
            )



conn = get_connection()
cursor = conn.cursor()

cursor.executemany(insert_query, matches)

conn.commit()

print("Inserted rows:", cursor.rowcount)

cursor.close()
conn.close()

cursor.close()
conn.close()

Inserted rows: 36


{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'WED, MAR 11 2026', 'matchScheduleList': [{'seriesName': 'Zimbabwe Women tour of New Zealand, 2026', 'matchInfo': [{'matchId': 122781, 'seriesId': 10245, 'matchDesc': '3rd ODI', 'matchFormat': 'ODI', 'startDate': '1773180000000', 'endDate': '1773273599000', 'team1': {'teamId': 98, 'teamName': 'New Zealand Women', 'teamSName': 'NZW', 'imageId': 776338}, 'team2': {'teamId': 388, 'teamName': 'Zimbabwe Women', 'teamSName': 'ZIMW', 'imageId': 776203}, 'venueInfo': {'ground': 'University Oval', 'city': 'Dunedin', 'country': 'New Zealand', 'timezone': '+13:00'}}], 'seriesId': 10245, 'seriesCategory': 'Women'}], 'longDate': '1773187200000'}}, {'adDetail': {'name': 'native_news_index_random_1', 'layout': 'native_large', 'position': 1}}, {'adDetail': {'name': 'native_news_index_random_1', 'layout': 'native_large', 'position': 1}}, {'scheduleAdWrapper': {'date': 'SUN, MAR 15 2026', 'matchScheduleList': [{'seriesName': 'South Africa Women tour o

{'matchScheduleMap': [{'scheduleAdWrapper': {'date': 'WED, MAR 11 2026',
    'matchScheduleList': [{'seriesName': 'Zimbabwe Women tour of New Zealand, 2026',
      'matchInfo': [{'matchId': 122781,
        'seriesId': 10245,
        'matchDesc': '3rd ODI',
        'matchFormat': 'ODI',
        'startDate': '1773180000000',
        'endDate': '1773273599000',
        'team1': {'teamId': 98,
         'teamName': 'New Zealand Women',
         'teamSName': 'NZW',
         'imageId': 776338},
        'team2': {'teamId': 388,
         'teamName': 'Zimbabwe Women',
         'teamSName': 'ZIMW',
         'imageId': 776203},
        'venueInfo': {'ground': 'University Oval',
         'city': 'Dunedin',
         'country': 'New Zealand',
         'timezone': '+13:00'}}],
      'seriesId': 10245,
      'seriesCategory': 'Women'}],
    'longDate': '1773187200000'}},
  {'adDetail': {'name': 'native_news_index_random_1',
    'layout': 'native_large',
    'position': 1}},
  {'adDetail': {'name': 'nat

In [1]:
# series and archives
import requests

series_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/international"
series_arc_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/international"

headers = {
    "x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# Current series
response1 = requests.get(series_url, headers=headers)
series = response1.json()

print("Current Series:")
print(series)

# Archived series
response2 = requests.get(series_arc_url, headers=headers)
series_archive = response2.json()

print("Archived Series:")
print(series_archive)

Current Series:
{'seriesMapProto': [{'date': 'FEBRUARY 2024', 'series': [{'id': 7572, 'name': 'ICC Cricket World Cup League Two 2023-27', 'startDt': '1707955200000', 'endDt': '1806364800000'}]}, {'date': 'FEBRUARY 2026', 'series': [{'id': 11253, 'name': "ICC Men's T20 World Cup 2026", 'startDt': '1770422400000', 'endDt': '1772928000000'}]}, {'date': 'MARCH 2026', 'series': [{'id': 11689, 'name': 'Bahrain tour of Malaysia, 2026', 'startDt': '1772841600000', 'endDt': '1773100800000'}, {'id': 11570, 'name': "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026", 'startDt': '1772928000000', 'endDt': '1773619200000'}, {'id': 11711, 'name': 'Lesotho tour of Botswana, 2026', 'startDt': '1773014400000', 'endDt': '1773532800000'}, {'id': 11584, 'name': 'Pakistan tour of Bangladesh, 2026', 'startDt': '1773187200000', 'endDt': '1773532800000'}, {'id': 11617, 'name': 'Afghanistan v Sri Lanka in UAE, 2026', 'startDt': '1773360000000', 'endDt': '1774396800000'}, {'id': 10234, 'name':

In [40]:
print(series.keys())
print(series_archive.keys())

dict_keys(['seriesMapProto', 'appIndex'])
dict_keys(['seriesMapProto', 'appIndex'])


In [41]:
print(len(series.get("seriesMapProto", [])))
print(len(series_archive.get("seriesMapProto", [])))

9
2


In [2]:
for month_data in series.get("seriesMapProto", []):
    print("MONTH BLOCK:", month_data)

MONTH BLOCK: {'date': 'FEBRUARY 2024', 'series': [{'id': 7572, 'name': 'ICC Cricket World Cup League Two 2023-27', 'startDt': '1707955200000', 'endDt': '1806364800000'}]}
MONTH BLOCK: {'date': 'FEBRUARY 2026', 'series': [{'id': 11253, 'name': "ICC Men's T20 World Cup 2026", 'startDt': '1770422400000', 'endDt': '1772928000000'}]}
MONTH BLOCK: {'date': 'MARCH 2026', 'series': [{'id': 11689, 'name': 'Bahrain tour of Malaysia, 2026', 'startDt': '1772841600000', 'endDt': '1773100800000'}, {'id': 11570, 'name': "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026", 'startDt': '1772928000000', 'endDt': '1773619200000'}, {'id': 11711, 'name': 'Lesotho tour of Botswana, 2026', 'startDt': '1773014400000', 'endDt': '1773532800000'}, {'id': 11584, 'name': 'Pakistan tour of Bangladesh, 2026', 'startDt': '1773187200000', 'endDt': '1773532800000'}, {'id': 11617, 'name': 'Afghanistan v Sri Lanka in UAE, 2026', 'startDt': '1773360000000', 'endDt': '1774396800000'}, {'id': 10234, 'name'

In [3]:
series_archive
series

{'seriesMapProto': [{'date': 'FEBRUARY 2024',
   'series': [{'id': 7572,
     'name': 'ICC Cricket World Cup League Two 2023-27',
     'startDt': '1707955200000',
     'endDt': '1806364800000'}]},
  {'date': 'FEBRUARY 2026',
   'series': [{'id': 11253,
     'name': "ICC Men's T20 World Cup 2026",
     'startDt': '1770422400000',
     'endDt': '1772928000000'}]},
  {'date': 'MARCH 2026',
   'series': [{'id': 11689,
     'name': 'Bahrain tour of Malaysia, 2026',
     'startDt': '1772841600000',
     'endDt': '1773100800000'},
    {'id': 11570,
     'name': "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026",
     'startDt': '1772928000000',
     'endDt': '1773619200000'},
    {'id': 11711,
     'name': 'Lesotho tour of Botswana, 2026',
     'startDt': '1773014400000',
     'endDt': '1773532800000'},
    {'id': 11584,
     'name': 'Pakistan tour of Bangladesh, 2026',
     'startDt': '1773187200000',
     'endDt': '1773532800000'},
    {'id': 11617,
     'name': 'Afghani

In [10]:
insert_current_query = """
INSERT INTO series_current (
    series_id, series_name, series_month,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
series_name = VALUES(series_name),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

insert_archieves_query = """
INSERT INTO series_archieves (
    series_id, series_name, series_month,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
series_name = VALUES(series_name),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

In [13]:

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in series.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_current_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (7572, 'ICC Cricket World Cup League Two 2023-27', 'FEBRUARY 2024', datetime.datetime(2024, 2, 15, 5, 30), datetime.datetime(2027, 3, 30, 5, 30))
Inserted: 2
VALUES: (11253, "ICC Men's T20 World Cup 2026", 'FEBRUARY 2026', datetime.datetime(2026, 2, 7, 5, 30), datetime.datetime(2026, 3, 8, 5, 30))
Inserted: 2
VALUES: (11689, 'Bahrain tour of Malaysia, 2026', 'MARCH 2026', datetime.datetime(2026, 3, 7, 5, 30), datetime.datetime(2026, 3, 10, 5, 30))
Inserted: 2
VALUES: (11570, "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026", 'MARCH 2026', datetime.datetime(2026, 3, 8, 5, 30), datetime.datetime(2026, 3, 16, 5, 30))
Inserted: 2
VALUES: (11711, 'Lesotho tour of Botswana, 2026', 'MARCH 2026', datetime.datetime(2026, 3, 9, 5, 30), datetime.datetime(2026, 3, 15, 5, 30))
Inserted: 2
VALUES: (11584, 'Pakistan tour of Bangladesh, 2026', 'MARCH 2026', datetime.datetime(2026, 3, 11, 5, 30), datetime.datetime(2026, 3, 15, 5, 30))
Inserted: 2
VALUES: (11617, 'Afghanista

In [15]:
cursor.execute("SELECT DATABASE();")
print("Connected DB:", cursor.fetchone())

Connected DB: ('cricbuzz_live',)


In [16]:

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in series_archive.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (11711, 'Lesotho tour of Botswana, 2026', '2026', datetime.datetime(2026, 3, 9, 5, 30), datetime.datetime(2026, 3, 15, 5, 30))
Inserted: 0
VALUES: (11570, "ICC Men's T20 2028 World Cup Americas Sub Regional Qualifier B, 2026", '2026', datetime.datetime(2026, 3, 8, 5, 30), datetime.datetime(2026, 3, 16, 5, 30))
Inserted: 0
VALUES: (11689, 'Bahrain tour of Malaysia, 2026', '2026', datetime.datetime(2026, 3, 7, 5, 30), datetime.datetime(2026, 3, 10, 5, 30))
Inserted: 0
VALUES: (11581, 'Kuwait tour of Hong Kong, China, 2026', '2026', datetime.datetime(2026, 2, 25, 5, 30), datetime.datetime(2026, 3, 1, 5, 30))
Inserted: 0
VALUES: (11542, 'Quadrangular T20I Series in Thailand 2026 ', '2026', datetime.datetime(2026, 2, 25, 5, 30), datetime.datetime(2026, 2, 28, 5, 30))
Inserted: 0
VALUES: (11528, 'Bahrain tour of Qatar 2026', '2026', datetime.datetime(2026, 2, 10, 5, 30), datetime.datetime(2026, 2, 15, 5, 30))
Inserted: 0
VALUES: (11253, "ICC Men's T20 World Cup 2026", '2026', datetim

In [18]:
# series and archives
import requests

series_arc_league_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/league"
series_arc_domestic_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/domestic"
series_arc_women_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/women"

headers = {
    "x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

# league series
response1 = requests.get(series_arc_league_url, headers=headers)
series_league = response1.json()

print("league series:")
print(series_league)

# domestic series
response2 = requests.get(series_arc_domestic_url, headers=headers)
series_domestic = response2.json()

print("domestic series:")
print(series_domestic)

# women series
response2 = requests.get(series_arc_women_url, headers=headers)
series_women = response2.json()

print("women series:")
print(series_women)

league series:
{'seriesMapProto': [{'date': '2026', 'series': [{'id': 11471, 'name': 'World Legends Pro T20 League 2026', 'startDt': '1769385600000', 'endDt': '1770163200000', 'thumborImageId': 836903}]}, {'date': '2025', 'series': [{'id': 11328, 'name': 'Bangladesh Premier League 2025-26', 'startDt': '1766707200000', 'endDt': '1769126400000', 'thumborImageId': 379130}, {'id': 10763, 'name': 'Super Smash 2025-26', 'startDt': '1766707200000', 'endDt': '1769817600000', 'thumborImageId': 379130}, {'id': 10394, 'name': 'SA20, 2025-26', 'startDt': '1766707200000', 'endDt': '1769299200000', 'thumborImageId': 835906}, {'id': 10289, 'name': 'Big Bash League 2025-26', 'startDt': '1765670400000', 'endDt': '1769299200000', 'thumborImageId': 798650}, {'id': 10884, 'name': 'International League T20, 2025-26', 'startDt': '1764633600000', 'endDt': '1767484800000', 'thumborImageId': 813363}, {'id': 11119, 'name': 'Abu Dhabi T10 League 2025', 'startDt': '1763424000000', 'endDt': '1764460800000', 'thumb

In [19]:
insert_archieves = """
INSERT INTO series_archieves (
    series_id, series_name, series_month,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
series_name = VALUES(series_name),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

In [20]:

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in series_league.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (11471, 'World Legends Pro T20 League 2026', '2026', datetime.datetime(2026, 1, 26, 5, 30), datetime.datetime(2026, 2, 4, 5, 30))
Inserted: 0
VALUES: (11328, 'Bangladesh Premier League 2025-26', '2025', datetime.datetime(2025, 12, 26, 5, 30), datetime.datetime(2026, 1, 23, 5, 30))
Inserted: 0
VALUES: (10763, 'Super Smash 2025-26', '2025', datetime.datetime(2025, 12, 26, 5, 30), datetime.datetime(2026, 1, 31, 5, 30))
Inserted: 0
VALUES: (10394, 'SA20, 2025-26', '2025', datetime.datetime(2025, 12, 26, 5, 30), datetime.datetime(2026, 1, 25, 5, 30))
Inserted: 0
VALUES: (10289, 'Big Bash League 2025-26', '2025', datetime.datetime(2025, 12, 14, 5, 30), datetime.datetime(2026, 1, 25, 5, 30))
Inserted: 0
VALUES: (10884, 'International League T20, 2025-26', '2025', datetime.datetime(2025, 12, 2, 5, 30), datetime.datetime(2026, 1, 4, 5, 30))
Inserted: 0
VALUES: (11119, 'Abu Dhabi T10 League 2025', '2025', datetime.datetime(2025, 11, 18, 5, 30), datetime.datetime(2025, 11, 30, 5, 30))
Ins

In [21]:

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in series_domestic.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (11553, 'CSA Provincial One-Day Challenge Division One 2026', '2026', datetime.datetime(2026, 2, 27, 5, 30), datetime.datetime(2026, 3, 29, 5, 30))
Inserted: 0
VALUES: (11559, 'CSA Provincial One-Day Challenge Division Two 2026', '2026', datetime.datetime(2026, 2, 21, 5, 30), datetime.datetime(2026, 3, 22, 5, 30))
Inserted: 0
VALUES: (11518, 'Pakistan A v England Lions in UAE, 2026', '2026', datetime.datetime(2026, 2, 20, 5, 30), datetime.datetime(2026, 3, 9, 5, 30))
Inserted: 0
VALUES: (11515, "ICC Men's T20 World Cup Warm up Matches 2026", '2026', datetime.datetime(2026, 2, 2, 5, 30), datetime.datetime(2026, 2, 6, 5, 30))
Inserted: 0
VALUES: (11209, 'ICC Under 19 World Cup 2026', '2026', datetime.datetime(2026, 1, 15, 5, 30), datetime.datetime(2026, 2, 6, 5, 30))
Inserted: 0
VALUES: (11394, 'ICC U19 World Cup Warm up Matches 2026', '2026', datetime.datetime(2026, 1, 9, 5, 30), datetime.datetime(2026, 1, 14, 5, 30))
Inserted: 0
VALUES: (11306, 'India Under-19 tour of South Afr

In [ ]:

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in series_domestic.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (10245, 'Zimbabwe Women tour of New Zealand, 2026', '2026', '1771977600000', '1773187200000')
VALUES: (11487, 'Sri Lanka Women tour of West Indies 2026', '2026', '1771545600000', '1772496000000')
VALUES: (9611, 'India Women tour of Australia, 2026', '2026', '1771113600000', '1772755200000')
VALUES: (11452, "ACC Women's Asia Cup Rising Stars 2026", '2026', '1770940800000', '1771718400000')
VALUES: (10146, 'Pakistan Women tour of South Africa, 2026', '2026', '1770681600000', '1772323200000')
VALUES: (11531, 'Oman Women tour of Qatar 2026', '2026', '1770595200000', '1771027200000')
VALUES: (11441, 'Denmark Women tour of Oman 2026', '2026', '1770076800000', '1770422400000')
VALUES: (11440, "Bhutan Women's T20I Tri-Series 2026", '2026', '1768867200000', '1769644800000')
VALUES: (11399, "ICC Women's T20 World Cup Global Qualifier 2026", '2026', '1768694400000', '1769904000000')
VALUES: (11275, "Women's Premier League 2026", '2026', '1767916800000', '1770249600000')
VALUES: (11262, 'S

In [22]:

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in series_women.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (10245, 'Zimbabwe Women tour of New Zealand, 2026', '2026', datetime.datetime(2026, 2, 25, 5, 30), datetime.datetime(2026, 3, 11, 5, 30))
Inserted: 0
VALUES: (11487, 'Sri Lanka Women tour of West Indies 2026', '2026', datetime.datetime(2026, 2, 20, 5, 30), datetime.datetime(2026, 3, 3, 5, 30))
Inserted: 0
VALUES: (9611, 'India Women tour of Australia, 2026', '2026', datetime.datetime(2026, 2, 15, 5, 30), datetime.datetime(2026, 3, 6, 5, 30))
Inserted: 0
VALUES: (11452, "ACC Women's Asia Cup Rising Stars 2026", '2026', datetime.datetime(2026, 2, 13, 5, 30), datetime.datetime(2026, 2, 22, 5, 30))
Inserted: 0
VALUES: (10146, 'Pakistan Women tour of South Africa, 2026', '2026', datetime.datetime(2026, 2, 10, 5, 30), datetime.datetime(2026, 3, 1, 5, 30))
Inserted: 0
VALUES: (11531, 'Oman Women tour of Qatar 2026', '2026', datetime.datetime(2026, 2, 9, 5, 30), datetime.datetime(2026, 2, 14, 5, 30))
Inserted: 0
VALUES: (11441, 'Denmark Women tour of Oman 2026', '2026', datetime.dateti

In [26]:
# series and archives
import requests

series_arc_inter_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/league"
series_arc_league_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/league"
series_arc_domestic_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/domestic"
series_arc_women_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/archives/women"

headers = {
    "x-rapidapi-key": "8a14d6f1femsh0a321ac05ddcde1p15c648jsn338a6e3092c8",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}


querystring = {"year":"2024"}

#international series
response1 = requests.get(series_arc_inter_url, headers=headers,params=querystring)
inter_2024 = response1.json()

print("international series:")
print(inter_2024)

# league series
response1 = requests.get(series_arc_league_url, headers=headers,params=querystring)
league_2024 = response1.json()

print("league series:")
print(league_2024)

# domestic series
response2 = requests.get(series_arc_domestic_url, headers=headers,params=querystring)
domestic_2024 = response2.json()

print("domestic series:")
print(domestic_2024)

# women series
response2 = requests.get(series_arc_women_url, headers=headers,params=querystring)
women_2024 = response2.json()

print("women series:")
print(women_2024)

international series:
{'seriesMapProto': [{'date': '2024', 'series': [{'id': 9201, 'name': 'Bangladesh Premier League, 2024-25', 'startDt': '1735516800000', 'endDt': '1738886400000', 'thumborImageId': 379130}, {'id': 8979, 'name': 'Super Smash 2024-25', 'startDt': '1735171200000', 'endDt': '1738454400000', 'thumborImageId': 379130}, {'id': 8535, 'name': 'Big Bash League 2024-25', 'startDt': '1734220800000', 'endDt': '1737936000000', 'thumborImageId': 379130}, {'id': 9225, 'name': 'Lanka T10 Super League, 2024', 'startDt': '1733875200000', 'endDt': '1734566400000', 'thumborImageId': 379130}, {'id': 9196, 'name': 'Global Super League, 2024', 'startDt': '1732579200000', 'endDt': '1733443200000', 'thumborImageId': 379130}, {'id': 9188, 'name': 'Abu Dhabi T10 League 2024', 'startDt': '1732147200000', 'endDt': '1733097600000', 'thumborImageId': 379130}, {'id': 8643, 'name': 'CSA T20 Challenge 2024', 'startDt': '1727395200000', 'endDt': '1729987200000', 'thumborImageId': 379130}, {'id': 8746,

In [24]:
insert_archieves = """
INSERT INTO series_archieves (
    series_id, series_name, series_month,
    start_date, end_date
)
VALUES (%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
series_name = VALUES(series_name),
start_date = VALUES(start_date),
end_date = VALUES(end_date)
"""

In [30]:
#series archieves 2024 league
from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in league_2024.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (9201, 'Bangladesh Premier League, 2024-25', '2024', datetime.datetime(2024, 12, 30, 5, 30), datetime.datetime(2025, 2, 7, 5, 30))
Inserted: 0
VALUES: (8979, 'Super Smash 2024-25', '2024', datetime.datetime(2024, 12, 26, 5, 30), datetime.datetime(2025, 2, 2, 5, 30))
Inserted: 0
VALUES: (8535, 'Big Bash League 2024-25', '2024', datetime.datetime(2024, 12, 15, 5, 30), datetime.datetime(2025, 1, 27, 5, 30))
Inserted: 0
VALUES: (9225, 'Lanka T10 Super League, 2024', '2024', datetime.datetime(2024, 12, 11, 5, 30), datetime.datetime(2024, 12, 19, 5, 30))
Inserted: 0
VALUES: (9196, 'Global Super League, 2024', '2024', datetime.datetime(2024, 11, 26, 5, 30), datetime.datetime(2024, 12, 6, 5, 30))
Inserted: 0
VALUES: (9188, 'Abu Dhabi T10 League 2024', '2024', datetime.datetime(2024, 11, 21, 5, 30), datetime.datetime(2024, 12, 2, 5, 30))
Inserted: 0
VALUES: (8643, 'CSA T20 Challenge 2024', '2024', datetime.datetime(2024, 9, 27, 5, 30), datetime.datetime(2024, 10, 27, 5, 30))
Inserted: 0

In [29]:
#series archieves 2024 international

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in inter_2024.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (9201, 'Bangladesh Premier League, 2024-25', '2024', datetime.datetime(2024, 12, 30, 5, 30), datetime.datetime(2025, 2, 7, 5, 30))
Inserted: 0
VALUES: (8979, 'Super Smash 2024-25', '2024', datetime.datetime(2024, 12, 26, 5, 30), datetime.datetime(2025, 2, 2, 5, 30))
Inserted: 0
VALUES: (8535, 'Big Bash League 2024-25', '2024', datetime.datetime(2024, 12, 15, 5, 30), datetime.datetime(2025, 1, 27, 5, 30))
Inserted: 0
VALUES: (9225, 'Lanka T10 Super League, 2024', '2024', datetime.datetime(2024, 12, 11, 5, 30), datetime.datetime(2024, 12, 19, 5, 30))
Inserted: 0
VALUES: (9196, 'Global Super League, 2024', '2024', datetime.datetime(2024, 11, 26, 5, 30), datetime.datetime(2024, 12, 6, 5, 30))
Inserted: 0
VALUES: (9188, 'Abu Dhabi T10 League 2024', '2024', datetime.datetime(2024, 11, 21, 5, 30), datetime.datetime(2024, 12, 2, 5, 30))
Inserted: 0
VALUES: (8643, 'CSA T20 Challenge 2024', '2024', datetime.datetime(2024, 9, 27, 5, 30), datetime.datetime(2024, 10, 27, 5, 30))
Inserted: 0

In [28]:
#series archieves 2024 domestic

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in domestic_2024.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (8357, 'Vijay Hazare Trophy 2024-25', '2024', datetime.datetime(2024, 12, 21, 5, 30), datetime.datetime(2025, 1, 18, 5, 30))
Inserted: 0
VALUES: (9221, 'ACC U19 Asia Cup, 2024', '2024', datetime.datetime(2024, 11, 29, 5, 30), datetime.datetime(2024, 12, 8, 5, 30))
Inserted: 0
VALUES: (8364, 'Syed Mushtaq Ali Trophy 2024', '2024', datetime.datetime(2024, 11, 23, 5, 30), datetime.datetime(2024, 12, 15, 5, 30))
Inserted: 0
VALUES: (9150, 'Sri Lanka A tour of Pakistan, 2024', '2024', datetime.datetime(2024, 11, 11, 5, 30), datetime.datetime(2024, 11, 29, 5, 30))
Inserted: 0
VALUES: (8939, 'Plunket Shield 2024-25', '2024', datetime.datetime(2024, 11, 11, 5, 30), datetime.datetime(2025, 4, 1, 5, 30))
Inserted: 0
VALUES: (9001, 'ICC CWC Challenge League B, 2024-26', '2024', datetime.datetime(2024, 11, 6, 5, 30), datetime.datetime(2026, 1, 30, 5, 30))
Inserted: 0
VALUES: (8276, 'India A tour of Australia, 2024', '2024', datetime.datetime(2024, 10, 31, 5, 30), datetime.datetime(2024, 11

In [ ]:
#series archieves 2024 women

from utils.db_connection import get_connection
from datetime import datetime

conn = get_connection()
cursor = conn.cursor()

for month_data in women_2024.get("seriesMapProto", []):

    series_month = month_data.get("date")

    for s in month_data.get("series", []):

        raw_start = s.get("startDt")
        raw_end = s.get("endDt")

        start_date = None
        end_date = None

        if raw_start:
            start_date = datetime.fromtimestamp(int(raw_start) / 1000)

        if raw_end:
            end_date = datetime.fromtimestamp(int(raw_end) / 1000)

        values = (
            s.get("id"),
            s.get("name"),
            series_month,
            start_date,
            end_date
        )

        print("VALUES:", values)

        cursor.execute(insert_archieves_query, values)
        print("Inserted:", cursor.rowcount)

conn.commit()

print("Series data inserted successfully")

VALUES: (9317, 'Philippines Women tour of Singapore, 2024', '2024', datetime.datetime(2024, 12, 23, 5, 30), datetime.datetime(2024, 12, 27, 5, 30))
Inserted: 0
VALUES: (9309, 'Myanmar Women tour of Bhutan, 2024', '2024', datetime.datetime(2024, 12, 21, 5, 30), datetime.datetime(2024, 12, 27, 5, 30))
Inserted: 0
VALUES: (8563, 'Australia Women tour of New Zealand, 2024', '2024', datetime.datetime(2024, 12, 19, 5, 30), datetime.datetime(2024, 12, 23, 5, 30))
Inserted: 0
VALUES: (9301, 'Jersey Women tour of Gibraltar, 2024', '2024', datetime.datetime(2024, 12, 15, 5, 30), datetime.datetime(2024, 12, 16, 5, 30))
Inserted: 0
VALUES: (9209, 'West Indies Women tour of India, 2024', '2024', datetime.datetime(2024, 12, 15, 5, 30), datetime.datetime(2024, 12, 27, 5, 30))
Inserted: 0
VALUES: (9293, 'Namibia Women tour of Malaysia, 2024', '2024', datetime.datetime(2024, 12, 10, 5, 30), datetime.datetime(2024, 12, 12, 5, 30))
Inserted: 0
VALUES: (7799, 'India Women tour of Australia, 2024', '2024',

In [20]:
#players india
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/teams/v1/2/players"

headers = {
	"x-rapidapi-key": "64ac7cdbacmshd400f941246fd04p1bd841jsn25ce69f2b2e4",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

print(response.json())
players_india = response.json()
players_india

{'player': [{'name': 'BATSMEN', 'imageId': 174146}, {'id': '11808', 'name': 'Shubman Gill', 'imageId': 616515, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm offbreak'}, {'id': '13940', 'name': 'Yashasvi Jaiswal', 'imageId': 591942, 'battingStyle': 'Left-hand bat', 'bowlingStyle': 'Right-arm legbreak'}, {'id': '13866', 'name': 'Sai Sudharsan', 'imageId': 717782, 'battingStyle': 'Left-hand bat', 'bowlingStyle': 'Right-arm legbreak'}, {'id': '576', 'name': 'Rohit Sharma', 'imageId': 616514, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm offbreak'}, {'id': '1413', 'name': 'Virat Kohli', 'imageId': 616517, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm medium'}, {'id': '7915', 'name': 'Suryakumar Yadav', 'imageId': 846028, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm offbreak'}, {'id': '9428', 'name': 'Shreyas Iyer', 'imageId': 616518, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm legbreak'}, {'id': '10896', 'name': '

{'player': [{'name': 'BATSMEN', 'imageId': 174146},
  {'id': '11808',
   'name': 'Shubman Gill',
   'imageId': 616515,
   'battingStyle': 'Right-hand bat',
   'bowlingStyle': 'Right-arm offbreak'},
  {'id': '13940',
   'name': 'Yashasvi Jaiswal',
   'imageId': 591942,
   'battingStyle': 'Left-hand bat',
   'bowlingStyle': 'Right-arm legbreak'},
  {'id': '13866',
   'name': 'Sai Sudharsan',
   'imageId': 717782,
   'battingStyle': 'Left-hand bat',
   'bowlingStyle': 'Right-arm legbreak'},
  {'id': '576',
   'name': 'Rohit Sharma',
   'imageId': 616514,
   'battingStyle': 'Right-hand bat',
   'bowlingStyle': 'Right-arm offbreak'},
  {'id': '1413',
   'name': 'Virat Kohli',
   'imageId': 616517,
   'battingStyle': 'Right-hand bat',
   'bowlingStyle': 'Right-arm medium'},
  {'id': '7915',
   'name': 'Suryakumar Yadav',
   'imageId': 846028,
   'battingStyle': 'Right-hand bat',
   'bowlingStyle': 'Right-arm offbreak'},
  {'id': '9428',
   'name': 'Shreyas Iyer',
   'imageId': 616518,
   'ba

In [24]:
insert_query = """
INSERT INTO players (player_id, name, playing_role, batting_style, bowling_style)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
name = VALUES(name),
playing_role = VALUES(playing_role),
batting_style = VALUES(batting_style),
bowling_style = VALUES(bowling_style)
"""

from utils.db_connection import get_connection

conn = get_connection()
cursor = conn.cursor()
players = players_india['player']   

current_role = None

for p in players:
    
    if 'id' not in p:
        current_role = p.get('name')
        continue

    values = (
        int(p['id']),
        p.get('name'),
        current_role,
        p.get('battingStyle'),
        p.get('bowlingStyle')
    )

    cursor.execute(insert_query, values)

conn.commit()

cursor.close()
conn.close()
print("values",values)

values (12926, 'Varun Chakaravarthy', 'BOWLER', 'Right-hand bat', 'Right-arm legbreak')


In [10]:
from concurrent.futures import ThreadPoolExecutor
import requests

team_ids = [2,3,4,5,6,7,8,9,10,11,12,13]

headers = {
    "x-rapidapi-key": "64ac7cdbacmshd400f941246fd04p1bd841jsn25ce69f2b2e4",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

def fetch(team_id):
    url = f"https://cricbuzz-cricket.p.rapidapi.com/teams/v1/{team_id}/players"
    data = requests.get(url, headers=headers).json()
    return {"team_id": team_id, "data": data}

with ThreadPoolExecutor() as executor:
    results = list(executor.map(fetch, team_ids))

print("Teams fetched:", len(results))

Teams fetched: 12


In [14]:
results

[{'team_id': 2,
  'data': {'player': [{'name': 'BATSMEN', 'imageId': 174146},
    {'id': '11808',
     'name': 'Shubman Gill',
     'imageId': 616515,
     'battingStyle': 'Right-hand bat',
     'bowlingStyle': 'Right-arm offbreak'},
    {'id': '13940',
     'name': 'Yashasvi Jaiswal',
     'imageId': 591942,
     'battingStyle': 'Left-hand bat',
     'bowlingStyle': 'Right-arm legbreak'},
    {'id': '13866',
     'name': 'Sai Sudharsan',
     'imageId': 717782,
     'battingStyle': 'Left-hand bat',
     'bowlingStyle': 'Right-arm legbreak'},
    {'id': '576',
     'name': 'Rohit Sharma',
     'imageId': 616514,
     'battingStyle': 'Right-hand bat',
     'bowlingStyle': 'Right-arm offbreak'},
    {'id': '1413',
     'name': 'Virat Kohli',
     'imageId': 616517,
     'battingStyle': 'Right-hand bat',
     'bowlingStyle': 'Right-arm medium'},
    {'id': '7915',
     'name': 'Suryakumar Yadav',
     'imageId': 846028,
     'battingStyle': 'Right-hand bat',
     'bowlingStyle': 'Right-ar

In [16]:
insert_query = """
INSERT INTO players (player_id, name, country, playing_role, batting_style, bowling_style)
VALUES (%s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
name = VALUES(name),
country = VALUES(country),
playing_role = VALUES(playing_role),
batting_style = VALUES(batting_style),
bowling_style = VALUES(bowling_style)
"""
team_player_query = """
INSERT IGNORE INTO team_players (team_id, player_id)
VALUES (%s, %s)
"""

from utils.db_connection import get_connection

team_map = {
    2: "India",
    3: "Pakistan",
    4: "Australia",
    5: "England",
    6: "Bangladesh",
    7: "Sri Lanka",
    8: "South Africa",
    9: "West Indies",
    10: "New Zealand",
    11: "Zimbabwe",
    12: "Ireland",
    13: "Afghanistan"
}

conn = get_connection()
cursor = conn.cursor()

for team_data in results:

    team_id = team_data["team_id"]
    country = team_map.get(team_id)

    players = team_data["data"].get("player", [])

    current_role = None

    for p in players:

        if "id" not in p:
            current_role = p.get("name")
            continue

        values = (
            int(p["id"]),
            p.get("name"),
            country,
            current_role,
            p.get("battingStyle"),
            p.get("bowlingStyle")
        )

        cursor.execute(insert_query, values)
        cursor.execute(team_player_query, (team_id, player_id))

conn.commit()

cursor.close()
conn.close()

print("Last inserted values:", values)

Last inserted values: (11178, 'Ben Sears', 'Afghanistan', 'BOWLER', 'Right-hand bat', 'Right-arm fast-medium')


In [4]:
for team in results:
    print(team)

{'player': [{'name': 'BATSMEN', 'imageId': 174146}, {'id': '8364', 'name': 'Imam-ul-Haq', 'imageId': 617489, 'battingStyle': 'Left-hand bat'}, {'id': '15847', 'name': 'Muhammad Hurraira', 'imageId': 174146, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm offbreak'}, {'id': '8359', 'name': 'Babar Azam', 'imageId': 846059, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm offbreak'}, {'id': '9545', 'name': 'Shan Masood', 'imageId': 244962, 'battingStyle': 'Left-hand bat', 'bowlingStyle': 'Right-arm medium'}, {'id': '10863', 'name': 'Fakhar Zaman', 'imageId': 846057, 'battingStyle': 'Left-hand bat', 'bowlingStyle': 'Left-arm orthodox'}, {'id': '12565', 'name': 'Khushdil Shah', 'imageId': 616448, 'battingStyle': 'Left-hand bat', 'bowlingStyle': 'Left-arm orthodox'}, {'id': '41113', 'name': 'Hasan Nawaz', 'imageId': 174146, 'battingStyle': 'Right-hand bat', 'bowlingStyle': 'Right-arm offbreak'}, {'id': '12139', 'name': 'Abdullah Shafique', 'imageId': 352418, 'batting

In [13]:
for team in results:
    print("Team ID:", team.get("teamId"))
    print("Players count:", len(team.get("player", [])))

Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0
Team ID: None
Players count: 0


In [50]:
#top 10 odi batsman
from concurrent.futures import ThreadPoolExecutor
import requests

player_ids = [25, 1413,104, 38, 102, 101,35, 213, 576, 29, 240, 1643,4160, 3838]

headers = {
    "x-rapidapi-key": "7cde9c129fmsh43ac0b206d50ebdp121f9cjsnfa25cdc2fd16",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

def fetch(pid):
    url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{pid}/batting"
    return requests.get(url, headers=headers).json()

with ThreadPoolExecutor() as executor:
    results = list(executor.map(fetch, player_ids))

print("Players fetched:", len(results))

Players fetched: 14


In [58]:
results

[{'message': 'Too many requests'},
 {'message': 'Too many requests'},
 {'message': 'Too many requests'},
 {'message': 'Too many requests'},
 {'message': 'Too many requests'},
 {'headers': ['ROWHEADER', 'Test', 'ODI', 'T20', 'IPL'],
  'values': [{'values': ['Matches', '149', '448', '55', '80']},
   {'values': ['Innings', '252', '418', '55', '78']},
   {'values': ['Runs', '11814', '12650', '1493', '1802']},
   {'values': ['Balls', '22959', '16019', '1121', '1462']},
   {'values': ['Highest', '374', '144', '100', '110']},
   {'values': ['Average', '49.85', '33.38', '31.77', '28.6']},
   {'values': ['SR', '51.46', '78.97', '133.19', '123.26']},
   {'values': ['Not Out', '15', '39', '8', '15']},
   {'values': ['Fours', '1387', '1119', '173', '200']},
   {'values': ['Sixes', '61', '76', '33', '39']},
   {'values': ['Ducks', '15', '28', '4', '2']},
   {'values': ['50s', '50', '77', '9', '10']},
   {'values': ['100s', '34', '19', '1', '1']},
   {'values': ['200s', '7', '0', '0', '0']},
   {'va

In [57]:
#odi highest


from utils.db_connection import get_connection

insert_query = """
INSERT INTO odi_player_stats
(player_id, player_name, matches, runs, batting_average, centuries, highest_score)

VALUES (%s,%s,%s,%s,%s,%s,%s)

ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
batting_average = VALUES(batting_average),
centuries = VALUES(centuries),
highest_score = VALUES(highest_score)
"""

conn = get_connection()
cursor = conn.cursor()

for pid, data in zip(player_ids, results):

    headers = data.get("headers")
    values = data.get("values")

    if not headers or not values:
        print("Skipped player:", pid, "Keys:", data.keys())
        continue

    if "ODI" not in headers:
        print("ODI stats missing:", pid)
        continue

    odi_index = headers.index("ODI")

    stats = {row["values"][0]: row["values"][odi_index] for row in values}

    player_name = data.get("appIndex", {}).get("seoTitle", "").split(" Profile")[0]

    values_insert = (
        pid,
        player_name,
        stats.get("Matches"),
        stats.get("Runs"),
        stats.get("Average"),
        stats.get("100s"),
        stats.get("Highest")
    )

    cursor.execute(insert_query, values_insert)

    print("Inserted:", values_insert)

conn.commit()
cursor.close()
conn.close()

print("All players processed successfully")

Skipped player: 25 Keys: dict_keys(['message'])
Skipped player: 1413 Keys: dict_keys(['message'])
Skipped player: 104 Keys: dict_keys(['message'])
Skipped player: 38 Keys: dict_keys(['message'])
Skipped player: 102 Keys: dict_keys(['message'])
Inserted: (101, 'Mahela Jayawardene', '448', '12650', '33.38', '19', '144')
Skipped player: 35 Keys: dict_keys(['message'])
Skipped player: 213 Keys: dict_keys(['message'])
Skipped player: 576 Keys: dict_keys(['message'])
Skipped player: 29 Keys: dict_keys(['message'])
Skipped player: 240 Keys: dict_keys(['message'])
Skipped player: 1643 Keys: dict_keys(['message'])
Skipped player: 4160 Keys: dict_keys(['message'])
Skipped player: 3838 Keys: dict_keys(['message'])
All players processed successfully


In [77]:
#test highest
from utils.db_connection import get_connection

insert_query = """
INSERT INTO test_player_stats
(player_id, player_name, runs, batting_average, centuries, highest_score)

VALUES (%s,%s,%s,%s,%s,%s)

ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
batting_average = VALUES(batting_average),
centuries = VALUES(centuries),
highest_score = VALUES(highest_score)
"""

conn = get_connection()
cursor = conn.cursor()

for pid, data in zip(player_ids, results):

    test_index = data["headers"].index("Test")

    stats = {row["values"][0]: row["values"][test_index] for row in data["values"]}

    player_name = data["appIndex"]["seoTitle"].split(" Profile")[0]

    hs = stats.get("Highest")

    values = (
    pid,
    player_name,
    stats.get("Runs"),
    stats.get("Average"),
    stats.get("100s"),
    stats.get("Highest")
)

    cursor.execute(insert_query, values)

    print("Inserted:", values)

conn.commit()
cursor.close()
conn.close()

print("All Test players processed successfully")

Inserted: (25, 'Sachin Tendulkar', '15921', '53.79', '51', '248')
Inserted: (1413, 'Virat Kohli', '9230', '46.85', '30', '254')
Inserted: (104, 'Kumar Sangakkara', '12400', '57.14', '38', '319')
Inserted: (38, 'Ricky Ponting', '13378', '51.85', '41', '257')
Inserted: (102, 'Sanath Jayasuriya', '6973', '40.07', '14', '340')
Inserted: (101, 'Mahela Jayawardene', '11814', '49.85', '34', '374')
Inserted: (35, 'Inzamam-ul-Haq', '8830', '49.33', '25', '329')
Inserted: (213, 'Jacques Kallis', '13289', '55.37', '45', '224')
Inserted: (576, 'Rohit Sharma', '4301', '40.58', '12', '212')
Inserted: (29, 'Sourav Ganguly', '7212', '42.18', '16', '239')
Inserted: (240, 'Brian Lara', '11953', '52.89', '34', '400')
Inserted: (1643, 'Aaron Finch', '278', '27.8', '0', '62')
Inserted: (4160, 'Sir Garry Sobers', '8032', '57.78', '26', '365')
Inserted: (3838, 'Kapil Dev', '5248', '31.05', '8', '163')
All Test players processed successfully


In [55]:
#T20 highest
from utils.db_connection import get_connection

insert_query = """
INSERT INTO t20i_player_stats
(player_id, player_name, runs, batting_average, centuries, highest_score)

VALUES (%s,%s,%s,%s,%s,%s)

ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
batting_average = VALUES(batting_average),
centuries = VALUES(centuries),
highest_score = VALUES(highest_score)
"""

conn = get_connection()
cursor = conn.cursor()

for pid, data in zip(player_ids, results):

    t20_index = data["headers"].index("T20")

    stats = {row["values"][0]: row["values"][t20_index] for row in data["values"]}

    player_name = data["appIndex"]["seoTitle"].split(" Profile")[0]

    hs = stats.get("Highest")

    values = (
    pid,
    player_name,
    stats.get("Runs"),
    stats.get("Average"),
    stats.get("100s"),
    stats.get("Highest")
)

    cursor.execute(insert_query, values)

    print("Inserted:", values)

conn.commit()
cursor.close()
conn.close()

print("All Test players processed successfully")

KeyError: 'headers'

In [60]:
#top 10 venue
from concurrent.futures import ThreadPoolExecutor
import requests

venue_ids = [50, 11, 15, 27, 87, 31, 51, 40, 29, 34]

headers = {
    "x-rapidapi-key": "99366df196mshc1d2db1859718f9p104787jsn6adf90f5eccc",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

def fetch(vid):
    url = f"https://cricbuzz-cricket.p.rapidapi.com/venues/v1/{vid}"
    
    try:
        response = requests.get(url, headers=headers)

        if response.status_code == 200 and response.text:
            return response.json()

        else:
            print(f"Venue {vid} error:", response.status_code)
            return None

    except requests.exceptions.RequestException as e:
        print("Request error:", e)
        return None


with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fetch, venue_ids))

# remove failed requests
results = [r for r in results if r is not None]

print("Venues fetched:", len(results))

Venues fetched: 10


In [53]:
#ALL rounder
from concurrent.futures import ThreadPoolExecutor
import requests

allrounder_ids = [213, 4160, 3838]

headers = {
    "x-rapidapi-key": "99366df196mshc1d2db1859718f9p104787jsn6adf90f5eccc",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

def fetch(aid):
    url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{aid}/bowling"
    return requests.get(url, headers=headers).json()

with ThreadPoolExecutor() as executor:
    results_1 = list(executor.map(fetch, allrounder_ids))

print("Players fetched:", len(results_1))

Players fetched: 3


In [61]:
results_1

[{'headers': ['ROWHEADER', 'Test', 'ODI', 'T20', 'IPL'],
  'values': [{'values': ['Matches', '166', '328', '25', '98']},
   {'values': ['Innings', '272', '283', '19', '89']},
   {'values': ['Balls', '20232', '10750', '276', '1742']},
   {'values': ['Runs', '9535', '8680', '333', '2293']},
   {'values': ['Maidens', '846', '78', '1', '3']},
   {'values': ['Wickets', '292', '273', '12', '65']},
   {'values': ['Avg', '32.65', '31.79', '27.75', '35.28']},
   {'values': ['Eco', '2.83', '4.84', '7.24', '7.9']},
   {'values': ['SR', '69.29', '39.38', '23.0', '26.8']},
   {'values': ['BBI', '6/54', '5/30', '4/15', '3/13']},
   {'values': ['BBM', '9/92', '5/30', '4/15', '3/13']},
   {'values': ['4w', '7', '2', '1', '0']},
   {'values': ['5w', '5', '2', '0', '0']},
   {'values': ['10w', '0', '0', '0', '0']}],
  'appIndex': {'seoTitle': 'Jacques Kallis Profile - Cricbuzz | Cricbuzz.com',
   'webURL': 'http://www.cricbuzz.com/profiles/213/jacques-kallis'},
  'seriesSpinner': [{'seriesName': 'Career

In [68]:
stats.get("Wickets")

'434'

Inserted: (25, 'Jacques Kallis', '8680', '273', '31.79', '4.84', '39.38')
Inserted: (25, 'Sir Garry Sobers', '31', '1', '31.0', '2.95', '63.0')
Inserted: (25, 'Kapil Dev', '6945', '253', '27.45', '3.72', '44.28')
All ODI bowling stats inserted successfully


In [72]:
from utils.db_connection import get_connection

insert_query = """
INSERT INTO odi_bowling_stats
(player_id, player_name, wickets)

VALUES (%s,%s,%s)

ON DUPLICATE KEY UPDATE
wickets = VALUES(wickets)
"""

conn = get_connection()
cursor = conn.cursor()

for aid, data in zip(allrounder_ids, results_1):

    odi_index = data["headers"].index("ODI")

    stats = {row["values"][0]: row["values"][odi_index] for row in data["values"]}

    player_name = data["appIndex"]["seoTitle"].split(" Profile")[0]

    wickets = stats.get("Wickets")

    values = (
        aid,
        player_name,
        wickets
    )

    cursor.execute(insert_query, values)

    print("Inserted:", values)

conn.commit()
cursor.close()
conn.close()

Inserted: (213, 'Jacques Kallis', '273')
Inserted: (4160, 'Sir Garry Sobers', '1')
Inserted: (3838, 'Kapil Dev', '253')


In [67]:
from utils.db_connection import get_connection

insert_query = """
INSERT INTO t20i_bowling_stats
(player_id, player_name, runs_conceded, wickets, bowling_average, economy, strike_rate)

VALUES (%s,%s,%s,%s,%s,%s,%s)

ON DUPLICATE KEY UPDATE
runs_conceded = VALUES(runs_conceded),
wickets = VALUES(wickets),
bowling_average = VALUES(bowling_average),
economy = VALUES(economy),
strike_rate = VALUES(strike_rate)
"""

conn = get_connection()
cursor = conn.cursor()

for bid, data in zip(player_ids, results_1):

    test_index = data["headers"].index("Test")

    stats = {row["values"][0]: row["values"][test_index] for row in data["values"]}

    player_name = data["appIndex"]["seoTitle"].split(" Profile")[0]

    values = (
        pid,
        player_name,
        stats.get("Runs"),
        stats.get("Wickets"),
        stats.get("Avg"),
        stats.get("Eco"),
        stats.get("SR")
    )

    cursor.execute(insert_query, values)

    print("Inserted:", values)

conn.commit()
cursor.close()
conn.close()

print("All ODI bowling stats inserted successfully")

Inserted: (25, 'Jacques Kallis', '9535', '292', '32.65', '2.83', '69.29')
Inserted: (25, 'Sir Garry Sobers', '7999', '235', '34.04', '2.33', '87.65')
Inserted: (25, 'Kapil Dev', '12867', '434', '29.65', '2.78', '63.92')
All ODI bowling stats inserted successfully


In [36]:
results

[{'ground': 'Narendra Modi Stadium',
  'city': 'Ahmedabad',
  'country': 'India',
  'timezone': '+05:30',
  'established': 1982,
  'capacity': '132,000',
  'knownAs': 'Motera Gujarat Stadium, Sardar Patel Stadium',
  'ends': 'Adani Pavilion End, GMDC End',
  'homeTeam': 'Gujarat, Gujarat Titans',
  'floodlights': True,
  'curator': 'Bagira Thakur',
  'profile': "<b>Introduction:</b>\n\nPopular as one of the premier cricket grounds in India, the stadium is generally known as the 'Motera Stadium' to avoid confusion with another stadium of the same name in the Navrangpura area. It is owned by the Gujarat Cricket Association and comes under the aegis of the West Zone.\n\n<b>Venue History:</b>\n\nBefore this stadium came into existence, international matches were played at the Ahmedabad Municipal Corporation's stadium, bearing the same name. However, in 1982, the Gujrat government donated a 50-acre land on the banks of the Sabarmati river and the Motera stadium was constructed in a span of 

In [41]:
import re
from utils.db_connection import get_connection

insert_query = """
    INSERT INTO venue (venueId, ground, city, country, capacity)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
    ground = VALUES(ground),
    city = VALUES(city),
    country = VALUES(country),
    capacity = VALUES(capacity)
    """

conn = get_connection()
cursor = conn.cursor()


for vid, data in zip(venue_ids, results):

    capacity = data.get("capacity")

    if capacity:
        capacity = int(re.search(r'\d+', capacity.replace(",", "")).group())

    values = (
        vid,
        data.get("ground"),
        data.get("city"),
        data.get("country"),
        capacity
    )

    cursor.execute(insert_query, values)

    # Debug statement
    print("Inserted:", values)

conn.commit()

cursor.close()
conn.close()

print("All venues processed successfully")

Inserted: (50, 'Narendra Modi Stadium', 'Ahmedabad', 'India', 132000)
Inserted: (11, 'MA Chidambaram Stadium', 'Chennai', 'India', 50000)
Inserted: (15, 'R.Premadasa Stadium', 'Colombo', 'Sri Lanka', 35000)
Inserted: (27, 'M.Chinnaswamy Stadium', 'Bengaluru', 'India', 40000)
Inserted: (87, 'Melbourne Cricket Ground', 'Melbourne', 'Australia', 100000)
Inserted: (31, 'Eden Gardens', 'Kolkata', 'India', 63000)
Inserted: (51, 'Arun Jaitley Stadium', 'Delhi', 'India', 48000)
Inserted: (40, 'Adelaide Oval', 'Adelaide', 'Australia', 53583)
Inserted: (29, 'Vidarbha Cricket Association Ground', 'Nagpur', 'India', 45000)
Inserted: (34, 'Sydney Cricket Ground', 'Sydney', 'Australia', 48000)
All venues processed successfully


In [2]:
import requests

match_id = 139489

url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

headers = {
    "x-rapidapi-key": "99366df196mshc1d2db1859718f9p104787jsn6adf90f5eccc",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)
data_1 = response.json()
data_1

{'scorecard': [{'inningsid': 1,
   'batsman': [{'id': 8271,
     'balls': 46,
     'runs': 89,
     'fours': 5,
     'sixes': 8,
     'strkrate': '193.48',
     'name': 'Sanju Samson',
     'nickname': 'Sanju Samson',
     'iscaptain': False,
     'iskeeper': True,
     'outdec': 'c (sub)Cole McConchie b James Neesham',
     'videotype': '',
     'videourl': '',
     'videoid': 0,
     'planid': 0,
     'imageid': 0,
     'premiumvideourl': '',
     'iscbplusfree': False,
     'ispremiumfree': False,
     'inmatchchange': '',
     'isoverseas': False,
     'playingxichange': ''},
    {'id': 12086,
     'balls': 21,
     'runs': 52,
     'fours': 6,
     'sixes': 3,
     'strkrate': '247.62',
     'name': 'Abhishek Sharma',
     'nickname': 'Abhishek Sharma',
     'iscaptain': False,
     'iskeeper': False,
     'outdec': 'c Tim Seifert b Rachin Ravindra',
     'videotype': '',
     'videourl': '',
     'videoid': 0,
     'planid': 0,
     'imageid': 0,
     'premiumvideourl': '',
     

In [3]:
from utils.db_connection import get_connection

insert_query = """
INSERT INTO batting_score
(match_id, team_id, innings_id, batting_position, player_id, player_name,
 runs, balls, fours, sixes, strike_rate, dismissal, is_captain, is_keeper)

VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
balls = VALUES(balls),
fours = VALUES(fours),
sixes = VALUES(sixes),
strike_rate = VALUES(strike_rate)
"""

conn = get_connection()
cursor = conn.cursor()

for innings in data_1["scorecard"]:

    innings_id = innings.get("inningsid")
    team_id = innings.get("batteamid")

    for pos, batsman in enumerate(innings["batsman"], start=1):

        strike_rate = batsman.get("strkrate")
        strike_rate = float(strike_rate) if strike_rate else None

        values = (
            match_id,
            team_id,
            innings_id,
            pos,
            batsman["id"],
            batsman["name"],
            batsman["runs"],
            batsman["balls"],
            batsman["fours"],
            batsman["sixes"],
            strike_rate,
            batsman["outdec"],
            batsman["iscaptain"],
            batsman["iskeeper"]
        )

        cursor.execute(insert_query, values)

conn.commit()
cursor.close()
conn.close()

print("Scorecard inserted successfully")

Scorecard inserted successfully


In [7]:
from utils.db_connection import get_connection

insert_query = """
INSERT INTO bowling_scorecard
(innings_id, player_id, player_name, overs, maidens, runs, wickets, economy)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
overs = VALUES(overs),
maidens = VALUES(maidens),
runs = VALUES(runs),
wickets = VALUES(wickets),
economy = VALUES(economy)
"""

conn = get_connection()
cursor = conn.cursor()

for innings in data_1["scorecard"]:

    innings_id = innings.get("inningsid")

    for bowler in innings["bowler"]:

        values = (
            innings_id,
            bowler["id"],
            bowler["name"],
            bowler["overs"],
            bowler["maidens"],
            bowler["runs"],
            bowler["wickets"],
            bowler["economy"]
        )

        cursor.execute(insert_query, values)

conn.commit()
cursor.close()
conn.close()

print("Bowling scorecard inserted successfully")

Bowling scorecard inserted successfully


In [2]:
q17=[138963, 139142, 139472, 139478, 139489]
len(q17)

5

In [7]:
toss

{'miniscore': {'batsmanstriker': {'id': 1448301,
   'balls': 1,
   'runs': 1,
   'fours': 0,
   'sixes': 0,
   'strkrate': '100',
   'name': 'Muhammad Arfan',
   'nickname': 'Muhammad Arfan',
   'iscaptain': False,
   'iskeeper': False,
   'outdec': '',
   'videotype': '',
   'videourl': '',
   'videoid': 0,
   'planid': 0,
   'imageid': 0,
   'premiumvideourl': '',
   'iscbplusfree': False,
   'ispremiumfree': False,
   'inmatchchange': '',
   'isoverseas': False,
   'playingxichange': ''},
  'batsmannonstriker': {'id': 22911,
   'balls': 53,
   'runs': 74,
   'fours': 6,
   'sixes': 3,
   'strkrate': '139.62264150943395',
   'name': 'Aryansh Sharma',
   'nickname': 'Aryansh Sharma',
   'iscaptain': False,
   'iskeeper': False,
   'outdec': '',
   'videotype': '',
   'videourl': '',
   'videoid': 0,
   'planid': 0,
   'imageid': 0,
   'premiumvideourl': '',
   'iscbplusfree': False,
   'ispremiumfree': False,
   'inmatchchange': '',
   'isoverseas': False,
   'playingxichange': ''},
 

In [3]:
import requests
from utils.db_connection import get_connection

conn = get_connection()
cursor = conn.cursor()

insert_query = """
INSERT INTO tossanalysis
(matchid, team1, team2, toss_winner, decision, match_winner)

VALUES (%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
team1 = VALUES(team1),
team2 = VALUES(team2),
toss_winner = VALUES(toss_winner),
decision = VALUES(decision),
match_winner = VALUES(match_winner)

"""

for ques_ana in q17:

    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{ques_ana}/leanback"

    headers = {
        "x-rapidapi-key": "f3b2ba43b4mshce68ad6a5f2a2d7p1a3701jsn0da633c136df",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    toss = response.json()

    if "matchheaders" not in toss:
        print("No matchheaders for:", ques_ana)
        continue

    header = toss["matchheaders"]

    team1 = header["team1"]["teamname"]
    team2 = header["team2"]["teamname"]

    toss_winner = header["tossresults"]["tosswinnername"]
    decision = header["tossresults"]["decision"]

    winning_team_id = header["winningteamid"]

    
    if winning_team_id == header["team1"]["teamid"]:
        match_winner = team1
    elif winning_team_id == header["team2"]["teamid"]:
        match_winner = team2
    else:
        match_winner = None

    values = (
        ques_ana,
        team1,
        team2,
        toss_winner,
        decision,
        match_winner
)

cursor.execute(insert_query, values)

print("Inserted:", values)


conn.commit()
cursor.close()
conn.close()

print("All toss data inserted successfully")

Inserted: (139489, 'India', 'New Zealand', 'New Zealand', 'Bowling', 'India')
All toss data inserted successfully


In [20]:
print(toss.keys())
print(toss["matchheaders"].keys())

dict_keys(['miniscore', 'matchheaders'])
dict_keys(['state', 'status', 'matchformat', 'matchstarttimestamp', 'teamdetails', 'momplayers', 'mosplayers', 'winningteamid', 'revisedtarget', 'matchendtimestamp', 'seriesid', 'matchdesc', 'seriesname', 'alerttype', 'tossresults', 'livestreamenabled', 'team1', 'team2'])


In [32]:
match_ids = [139478, 139489, 139461, 139426, 139381, 139307, 139216, 139129, 138974, 145475]

In [33]:
import requests
import time

for match_id in match_ids:

    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

    headers = {
        "x-rapidapi-key": "f3b2ba43b4mshce68ad6a5f2a2d7p1a3701jsn0da633c136df",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com",
        "Content-Type": "application/json"
    }


    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        print(match_id, "Scorecard loaded")

    elif response.status_code == 410:
        print(match_id, "Scorecard not available")
        continue

    else:
        print(match_id, "Error:", response.status_code)

    time.sleep(2)

139478 Scorecard loaded
139489 Scorecard loaded
139461 Scorecard loaded
139426 Scorecard loaded
139381 Scorecard loaded
139307 Scorecard loaded
139216 Scorecard loaded
139129 Scorecard loaded
138974 Scorecard loaded
145475 Scorecard loaded


In [36]:
import requests
from utils.db_connection import get_connection
import time

conn = get_connection()
cursor = conn.cursor()

insert_query = """
INSERT INTO bowling_facts
(match_id, innings_id, player_id, player_name, overs, maidens, runs, wickets, economy)

VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)

ON DUPLICATE KEY UPDATE
overs = VALUES(overs),
maidens = VALUES(maidens),
runs = VALUES(runs),
wickets = VALUES(wickets),
economy = VALUES(economy)
"""

for match_id in match_ids:

    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

    headers = {
        "x-rapidapi-key": "f3b2ba43b4mshce68ad6a5f2a2d7p1a3701jsn0da633c136df",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(match_id, "Skipped:", response.status_code)
        continue

    data = response.json()

    scorecard = data.get("scorecard", [])

    for innings in scorecard:

        innings_id = innings.get("inningsId") or innings.get("inningsid")

        for bowler in innings.get("bowler", []):

            values = (
                match_id,
                innings_id,
                bowler.get("id"),
                bowler.get("name"),
                bowler.get("overs"),
                bowler.get("maidens"),
                bowler.get("runs"),
                bowler.get("wickets"),
                bowler.get("economy")
            )

            cursor.execute(insert_query, values)

    print("Inserted match:", match_id)

    time.sleep(1)   # prevents API rate limit

conn.commit()
cursor.close()
conn.close()

print("All bowling data inserted successfully")

Inserted match: 139478
Inserted match: 139489
Inserted match: 139461
Inserted match: 139426
Inserted match: 139381
Inserted match: 139307
Inserted match: 139216
Inserted match: 139129
Inserted match: 138974
Inserted match: 145475
All bowling data inserted successfully


In [44]:
import requests
import re
import time
from utils.db_connection import get_connection

match_ids = [56955, 56962, 56964, 56969, 56976, 49488, 49495, 49502, 49509, 49514, 49521, 50942, 50944, 50949, 50951, 50921, 50928,
50935, 49857, 49864, 49871, 48396, 48401, 48408, 48415, 48424, 48431, 45726, 45731, 48016, 41846, 48023, 77856, 77863,
77870, 77873, 77880, 77887, 77894, 77898, 82441, 82448, 82455]


insert_query = """
INSERT INTO batting_facts
(match_id, innings_id, player_id, player_name, runs, balls, fours, sixes, strike_rate, match_year)

VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)

ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
balls = VALUES(balls),
fours = VALUES(fours),
sixes = VALUES(sixes),
strike_rate = VALUES(strike_rate),
match_year = VALUES(match_year)
"""

conn = get_connection()
cursor = conn.cursor()

for match_id in match_ids:

    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

    headers = {
        "x-rapidapi-key": "f3b2ba43b4mshce68ad6a5f2a2d7p1a3701jsn0da633c136df",
        "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Skipped:", match_id)
        continue

    data = response.json()

    scorecard = data.get("scorecard", [])

    seo_title = data["appindex"]["seotitle"]
    
    year_match = re.search(r"\b(20\d{2})\b", seo_title)
    match_year = int(year_match.group(1)) if year_match else None

    for innings in scorecard:

        innings_id = innings.get("inningsId") or innings.get("inningsid")

        for batsman in innings.get("batsman", []):
            strike_rate = batsman.get("strkrate")

            try:
                strike_rate = float(strike_rate)
            except:
                strike_rate = None

            values = (
                match_id,
                innings_id,
                batsman.get("id"),
                batsman.get("name"),
                batsman.get("runs"),
                batsman.get("balls"),
                batsman.get("fours"),
                batsman.get("sixes"),
                
                strike_rate,
                match_year
                
            )

            cursor.execute(insert_query, values)

    print("Inserted match:", match_id)

    time.sleep(1)  # avoid API rate limit

conn.commit()
cursor.close()
conn.close()

print("Batting facts inserted successfully")

Inserted match: 56955
Inserted match: 56962
Inserted match: 56964
Inserted match: 56969
Inserted match: 56976
Inserted match: 49488
Inserted match: 49495
Inserted match: 49502
Inserted match: 49509
Inserted match: 49514
Inserted match: 49521
Inserted match: 50942
Inserted match: 50944
Inserted match: 50949
Inserted match: 50951
Inserted match: 50921
Inserted match: 50928
Inserted match: 50935
Inserted match: 49857
Inserted match: 49864
Inserted match: 49871
Inserted match: 48396
Inserted match: 48401
Inserted match: 48408
Inserted match: 48415
Inserted match: 48424
Inserted match: 48431
Inserted match: 45726
Inserted match: 45731
Inserted match: 48016
Inserted match: 41846
Inserted match: 48023
Inserted match: 77856
Inserted match: 77863
Inserted match: 77870
Inserted match: 77873
Inserted match: 77880
Inserted match: 77887
Inserted match: 77894
Inserted match: 77898
Inserted match: 82441
Inserted match: 82448
Inserted match: 82455
Batting facts inserted successfully


In [59]:
import requests
import time
from utils.db_connection import get_connection

match_ids = [77863,77870,77873,77880,77887,77894,77898,82441,82448,82455]

headers = {
    "x-rapidapi-key": "7cde9c129fmsh43ac0b206d50ebdp121f9cjsnfa25cdc2fd16",
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

insert_query = """
INSERT INTO batting_scorecard
(innings_id, player_id, player_name, runs, balls, fours, sixes, strike_rate, dismissal, is_captain, is_keeper, team_name)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
runs = VALUES(runs),
balls = VALUES(balls),
fours = VALUES(fours),
sixes = VALUES(sixes),
strike_rate = VALUES(strike_rate)
"""

conn = get_connection()
cursor = conn.cursor()

for match_id in match_ids:

    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Error:", match_id)
        continue

    data = response.json()

    scorecard = data.get("scorecard", [])

    for innings in scorecard:

        innings_id = innings.get("inningsid")

        team_name = innings.get("batteamname")

        for batsman in innings.get("batsman", []):

            strike_rate = batsman.get("strkrate")
            strike_rate = float(strike_rate) if strike_rate else None

            values = (
                innings_id,
                batsman["id"],
                batsman["name"],
                batsman["runs"],
                batsman["balls"],
                batsman["fours"],
                batsman["sixes"],
                strike_rate,
                batsman.get("outdec"),
                batsman.get("iscaptain"),
                batsman.get("iskeeper"),
                team_name
            )

            cursor.execute(insert_query, values)

    print("Inserted match:", match_id)

    time.sleep(1)  # prevent API rate limit

conn.commit()
cursor.close()
conn.close()

print("All batting scorecards inserted successfully")

Inserted match: 77863
Inserted match: 77870
Inserted match: 77873
Inserted match: 77880
Inserted match: 77887
Inserted match: 77894
Inserted match: 77898
Inserted match: 82441
Inserted match: 82448
Inserted match: 82455
All batting scorecards inserted successfully


In [1]:
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/stats/v1/topstats/0?statsType=mostRuns"

headers = {
	"x-rapidapi-key": "7cde9c129fmsh43ac0b206d50ebdp121f9cjsnfa25cdc2fd16",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

mostruns = response.json()
mostruns

{'filter': {'matchtype': [{'matchTypeId': '1', 'matchTypeDesc': 'test'}, {'matchTypeId': '2', 'matchTypeDesc': 'odi'}, {'matchTypeId': '3', 'matchTypeDesc': 't20i'}], 'team': [{'id': '2', 'teamShortName': 'IND'}, {'id': '27', 'teamShortName': 'IRE'}, {'id': '3', 'teamShortName': 'PAK'}, {'id': '4', 'teamShortName': 'AUS'}, {'id': '5', 'teamShortName': 'SL'}, {'id': '6', 'teamShortName': 'BAN'}, {'id': '9', 'teamShortName': 'ENG'}, {'id': '10', 'teamShortName': 'WI'}, {'id': '11', 'teamShortName': 'RSA'}, {'id': '12', 'teamShortName': 'ZIM'}, {'id': '13', 'teamShortName': 'NZ'}, {'id': '96', 'teamShortName': 'AFG'}], 'selectedMatchType': 'test'}, 'headers': ['Batter', 'M', 'I', 'R', 'Avg'], 'values': [{'values': ['25', 'Tendulkar', '200', '329', '15921', '53.79']}, {'values': ['8019', 'Root', '163', '298', '13943', '51.07']}, {'values': ['38', 'R Ponting', '168', '287', '13378', '51.85']}, {'values': ['213', 'Kallis', '166', '280', '13289', '55.37']}, {'values': ['27', 'Dravid', '164', 

{'filter': {'matchtype': [{'matchTypeId': '1', 'matchTypeDesc': 'test'},
   {'matchTypeId': '2', 'matchTypeDesc': 'odi'},
   {'matchTypeId': '3', 'matchTypeDesc': 't20i'}],
  'team': [{'id': '2', 'teamShortName': 'IND'},
   {'id': '27', 'teamShortName': 'IRE'},
   {'id': '3', 'teamShortName': 'PAK'},
   {'id': '4', 'teamShortName': 'AUS'},
   {'id': '5', 'teamShortName': 'SL'},
   {'id': '6', 'teamShortName': 'BAN'},
   {'id': '9', 'teamShortName': 'ENG'},
   {'id': '10', 'teamShortName': 'WI'},
   {'id': '11', 'teamShortName': 'RSA'},
   {'id': '12', 'teamShortName': 'ZIM'},
   {'id': '13', 'teamShortName': 'NZ'},
   {'id': '96', 'teamShortName': 'AFG'}],
  'selectedMatchType': 'test'},
 'headers': ['Batter', 'M', 'I', 'R', 'Avg'],
 'values': [{'values': ['25', 'Tendulkar', '200', '329', '15921', '53.79']},
  {'values': ['8019', 'Root', '163', '298', '13943', '51.07']},
  {'values': ['38', 'R Ponting', '168', '287', '13378', '51.85']},
  {'values': ['213', 'Kallis', '166', '280', '132

In [5]:
import time
from utils.db_connection import get_connection

insert_query = """
 INSERT INTO most_run_stats
    (player_id, player_name, matches, innings, runs, average)
    VALUES (%s,%s,%s,%s,%s,%s)
    ON DUPLICATE KEY UPDATE
        player_name=VALUES(player_name),
        matches=VALUES(matches),
        innings=VALUES(innings),
        runs=VALUES(runs),
        average=VALUES(average),
        
"""

conn = get_connection()
cursor = conn.cursor()

data = mostruns  

count = 0

#  LOOP
for player in data.get("values", [])[:20]:

    row = player["values"]

    try:
        values = (
            int(row[0]),   
            row[1],        
            int(row[2]),   
            int(row[3]),   
            int(row[4]),   
            float(row[5]),  
            
        )

        cursor.execute(insert_query, values)

        count += 1

    except Exception as e:
        print(f"Error inserting {row[1]}:", e)

print(f"Inserted Top {count} players")

conn.commit()
cursor.close()
conn.close()

Error inserting Tendulkar: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '' at line 9
Error inserting Root: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '' at line 9
Error inserting R Ponting: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '' at line 9
Error inserting Kallis: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '' at line 9
Error inserting Dravid: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '' at line 9
Error inserting Cook: 1064 (42000): You have an error in your SQL synta